# Audited GNN + BERT Music Context — Full Inline / Drive-Persistent / Fail-Fast Colab

This notebook is **self-contained**: the former cells that only called `src/*.py` or `scripts/*.sh` have been replaced with the actual Python implementation. It uses the **full available MusicCaps and GTZAN data** (no artificial row/sample limit), trains the graph encoder rather than caching a random frozen graph representation, and **fine-tunes DistilBERT end-to-end by default on GPU**.

The experimental rules remain strict: MusicCaps captions/tags are paired only with audio from the same `ytid`; GTZAN is a separate audio-only genre experiment; train/validation/test partitions are disjoint; model selection and threshold calibration use validation only; test metrics are computed only after selection.

The notebook is intentionally verbose. It logs dataset coverage, class support, epoch losses, validation metrics, learning rates, GPU memory, and produces graph/audio/training/retrieval visualizations throughout the run.


**V3 persistence update:** Google Drive is mounted before data/model initialization. MusicCaps targets the 766 usable clips, pre-filters known rejected YouTube IDs, persistently caches new failures, and immediately saves validation-best training checkpoints to Drive.


In [ ]:
# Cell 1 — Install dependencies, mount Google Drive, and initialize a persistent high-throughput runtime.
# IMPORTANT: all datasets, processed graphs, checkpoints, metrics, plots, logs, and model caches live in Drive.
import sys, subprocess, importlib.util, os, platform, json, math, time, random, ast, csv, shutil, tarfile, urllib.request, re
from pathlib import Path

REQUIRED = {
    'torch_geometric': 'torch-geometric>=2.5',
    'transformers': 'transformers>=4.40',
    'librosa': 'librosa>=0.10',
    'soundfile': 'soundfile>=0.12',
    'sklearn': 'scikit-learn>=1.3',
    'pandas': 'pandas>=2.0',
    'matplotlib': 'matplotlib>=3.7',
    'yaml': 'pyyaml>=6.0',
    'tqdm': 'tqdm>=4.66',
    'yt_dlp': 'yt-dlp>=2025.1.0',
    'iterstrat': 'iterative-stratification>=0.1.9',
    'networkx': 'networkx>=3.0',
    'psutil': 'psutil>=5.9',
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('[SETUP] Installing:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('[SETUP] All required packages are already installed.')

# --------------------------- PERSISTENT GOOGLE DRIVE ROOT ---------------------------
IN_COLAB = importlib.util.find_spec('google.colab') is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/CSE425_GNN_BERT_Music_Context')
else:
    DRIVE_ROOT = Path.cwd() / 'CSE425_GNN_BERT_Music_Context'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
ROOT = DRIVE_ROOT
os.chdir(ROOT)  # Make every legacy relative save path Drive-backed too.

# Persist pretrained-model caches too, so session termination does not force another model download.
os.environ['HF_HOME'] = str(ROOT/'model_cache/huggingface')
os.environ['TORCH_HOME'] = str(ROOT/'model_cache/torch')
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
Path(os.environ['TORCH_HOME']).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import psutil
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch_geometric.data import Data, Batch
from torch_geometric.nn import SAGEConv, global_mean_pool
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, average_precision_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import librosa

for p in ['data/raw/musiccaps_audio', 'data/raw/gtzan', 'data/processed/musiccaps_graphs',
          'data/processed/gtzan_graphs', 'data/processed/gtzan_mels', 'data/splits',
          'results/plots', 'results/run_cache', 'checkpoints', 'checkpoints/resume', 'model_cache']:
    (ROOT/p).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CPU_COUNT = os.cpu_count() or 2
RAM_GB = psutil.virtual_memory().total / 2**30
if DEVICE.type == 'cuda':
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try: torch.set_float32_matmul_precision('high')
    except Exception: pass
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    GPU_NAME, VRAM_GB, AMP_DTYPE = 'CPU', 0.0, torch.float32

# MusicCaps: use the full KNOWN-USABLE volume from the previous run, but stop wasting time after 766 usable pairs.
MUSICCAPS_TARGET_USABLE = 766
TEXT_MODEL_NAME = 'distilbert-base-uncased'
TEXT_MAX_LENGTH = 128
FULL_TEXT_FINETUNE = True
SEEDS = [17, 42, 1337]
SPLIT_SEED = 425
TOP_K_TAGS = 50
MIN_TAG_COUNT = 8
EPOCHS = 40
PATIENCE = 7
AUDIO_SR = 22050
CLIP_SECONDS = 10.0
SEGMENT_SECONDS = 2.0
SEGMENT_HOP_SECONDS = 0.5
N_MELS = 64
N_MFCC = 20
GRAPH_TOP_K = 4
GRAPH_MIN_SIM = 0.55

# Fail fast on rejected YouTube videos. Known permanent failures are filtered before spawning yt-dlp.
YTDLP_TIMEOUT_SECONDS = 50
YTDLP_SOCKET_TIMEOUT = 8
YTDLP_RETRIES = 0

# Scale batch/model size to actual VRAM instead of imposing a tiny batch.
if VRAM_GB >= 30:
    TRAIN_BS, EVAL_BS, CONTRASTIVE_BS = 128, 256, 192
    GRAPH_HIDDEN, FUSION_DIM, ATTN_HEADS = 512, 512, 8
elif VRAM_GB >= 14:
    TRAIN_BS, EVAL_BS, CONTRASTIVE_BS = 64, 128, 96
    GRAPH_HIDDEN, FUSION_DIM, ATTN_HEADS = 384, 384, 8
elif VRAM_GB >= 8:
    TRAIN_BS, EVAL_BS, CONTRASTIVE_BS = 32, 64, 48
    GRAPH_HIDDEN, FUSION_DIM, ATTN_HEADS = 256, 256, 8
else:
    TRAIN_BS, EVAL_BS, CONTRASTIVE_BS = 8, 16, 16
    GRAPH_HIDDEN, FUSION_DIM, ATTN_HEADS = 192, 192, 4
GRAPH_BATCH = max(64, TRAIN_BS * 4)
CNN_BATCH = max(16, TRAIN_BS)
NUM_WORKERS = min(8, max(2, CPU_COUNT // 2))
DOWNLOAD_WORKERS = min(10, max(3, CPU_COUNT // 2))

print('='*96)
print('[RUNTIME] Persistent high-throughput configuration')
print('='*96)
print(f'Persistent project root: {ROOT}')
print(f'Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}')
print(f'Device: {DEVICE} | GPU: {GPU_NAME} | VRAM: {VRAM_GB:.2f} GB | AMP dtype: {AMP_DTYPE}')
print(f'CPU threads: {CPU_COUNT} | RAM: {RAM_GB:.1f} GB | DataLoader workers: {NUM_WORKERS}')
print(f'Train/Eval/Contrastive batch: {TRAIN_BS}/{EVAL_BS}/{CONTRASTIVE_BS}')
print(f'Graph hidden: {GRAPH_HIDDEN} | Fusion dim: {FUSION_DIM} | Full DistilBERT fine-tune: {FULL_TEXT_FINETUNE}')
print(f'MusicCaps target usable clips: {MUSICCAPS_TARGET_USABLE}')
print('[PERSISTENCE] Runtime termination will NOT delete Drive-backed audio, graphs, checkpoints, metrics, or logs.')


In [ ]:
# Cell 2 — Reproducibility, persistent logging, GPU-memory reporting, JSON helpers.
from datetime import datetime

RUN_LOG_PATH = ROOT/'results/runtime.log'

def seed_everything(seed: int):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def gpu_mem_string():
    if DEVICE.type != 'cuda': return 'CPU'
    alloc = torch.cuda.memory_allocated()/2**30
    reserv = torch.cuda.memory_reserved()/2**30
    peak = torch.cuda.max_memory_allocated()/2**30
    return f'GPU alloc={alloc:.2f}G reserved={reserv:.2f}G peak={peak:.2f}G/{VRAM_GB:.1f}G'


def log(msg, tag='INFO'):
    line=f'[{datetime.now().strftime("%H:%M:%S")}] [{tag}] {msg}'
    print(line)
    try:
        with RUN_LOG_PATH.open('a',encoding='utf-8') as f: f.write(line+'\n')
    except Exception:
        pass


def write_json(obj, path):
    path = Path(path)
    if not path.is_absolute(): path = ROOT/path
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    with tmp.open('w', encoding='utf-8') as f: json.dump(obj, f, indent=2)
    tmp.replace(path)


def save_current_figure(name):
    out=ROOT/'results/plots'/name
    out.parent.mkdir(parents=True,exist_ok=True)
    plt.savefig(out,dpi=220,bbox_inches='tight')
    log(f'Saved plot: {out}', 'PLOT')

def safe_loader_kwargs(shuffle=False):
    kw = dict(shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
    if NUM_WORKERS > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    return kw

seed_everything(SPLIT_SEED)
log(f'Reproducibility helpers initialized. Persistent log: {RUN_LOG_PATH}')

## MusicCaps — full available paired audio + caption/tag experiment

Every graph, caption and tag vector below comes from the same MusicCaps `ytid`. The downloader attempts **all rows** and resumes existing clips. Removed/private/blocked videos are logged as missing; they are never replaced with audio from another dataset.


In [ ]:
# Cell 3 — Persistent, fail-fast MusicCaps downloader targeting the 766 known-usable pairs.
# Known rejected IDs are removed BEFORE yt-dlp. Every new permanent failure is cached in Drive and skipped forever after.
import threading
failure_cache_lock=threading.Lock()

MUSICCAPS_META = ROOT/'data/raw/musiccaps-public.csv'
MUSICCAPS_AUDIO = ROOT/'data/raw/musiccaps_audio'
USER_BLOCKLIST_PATH = ROOT/'data/splits/musiccaps_blocked_ids_user.txt'
FAILED_CACHE_PATH = ROOT/'data/splits/musiccaps_failed_cache.json'

if not MUSICCAPS_META.exists():
    url = 'https://huggingface.co/datasets/google/MusicCaps/resolve/main/musiccaps-public.csv'
    log(f'Downloading MusicCaps metadata: {url}', 'DOWNLOAD')
    urllib.request.urlretrieve(url, MUSICCAPS_META)
else:
    log(f'Using existing MusicCaps metadata: {MUSICCAPS_META}', 'DOWNLOAD')

df_mc_raw = pd.read_csv(MUSICCAPS_META)
required = {'ytid','caption','aspect_list'}
missing_cols = required - set(df_mc_raw.columns)
assert not missing_cols, f'MusicCaps metadata missing columns: {missing_cols}'
df_mc_raw['ytid']=df_mc_raw['ytid'].astype(str).str.replace('\\_','_',regex=False)
log(f'MusicCaps metadata rows: {len(df_mc_raw):,}', 'DATA')

# IDs supplied from your blocked-video extraction. This seed is merged with the persistent Drive list.
BLOCKED_SEED_TEXT = r"""-0xzrMun0Rs
-FlvaZQOr2I
-QuWdnmn-kM
-kpR93atgd8
-sevczF5etI
00M9FhCet6s
03frQGyrgQ4
0AhX-ReXovI
0J_2K1Gvruk
0KCVgexi4yU
0LLlcPiatiU
0QYNC7J05XI
0RcMzUdXDRQ
0VjPCd62oKg
0VsjSa1X7iA
0i8VM_EooCs
0khKvVDyYV4
1RhYdQnZ_hw
1SO5RJLWKAs
1TyOPtg0Yfk
1W2FOzSXsxs
1YPYQP6yupA
1Ziku4FLka4
1i1sbQOILb0
2-K-7T8ZIWA
25Ccp8usBtE
25RWrqQol7Y
2G5bSYHcJSM
2KGnpMYHBGI
2ZrqWkdwVzo
2hgvuYGc95o
2xATintzaj4
2zrPFxxT1VM
32C6w8V7TX8
3EXXs3x4Ius
3TO4C7SiC7I
3VEMHWnewuc
3obJKn19jTE
4--05CAaDsg
42xehaGoJbc
4HfU5OQUqq0
4Psyk_xyBl0
4VbXYuCz4M0
4q6e_ZDFOZI
4vWChPYkuwA
4ymXDU-48EE
577NM64YL18
5XXAeSybGK0
5Y_mT93tkvQ
5pIdH6p3kuo
63rqIYPHvlc
6UJhTZgnVro
6_zzu2KON6c
71hqRT9U0wg
73YTz8RC2Fo
78BtX0oNXHQ
7B1OAtD_VIA
7EvLwfwRrqA
7LGLhQFiE0s
7nrOZXbpXBo
8FZb_R2UANY
8Y0FRjkX9xI
8ZK1ajW598M
8fibq2VXibw
8hT2G2ew6-c
8olHAhUKkuk
98YBS2tdpdU
9DCJTAzUwNc
9ZeoYezrI7Q
A-oSBMP-Zy4
A2bI-MIIJfU
A9ECxUIqw6s
AAWe4zRLVVU
AFWy1qyyMHE
APUDprnfOIs
AaUZb-iRStE
AeXDtbfpQlQ
Ah_aYOGnQ_I
AtJ2RXQ98kY
B1vY7kxQ9Xg
B4KIQtk7fT4
B7iRvj8y9aU
BKQYrrJVg6g
BMPmavj4keA
Bd0PbyrG6H4
BeFzozm_H5M
BfUoopDpmmY
BiQik0xsWxk
BsVsoZ4ojp0
C33WdI64FiY
C6-JxDWYJ-A
C7OIuhWSbjU
CCFYOw8keiI
CN2QSmhP-HI
CaCjiFUL6Fg
CtcmYF7mufk
CuUu6L5hhMs
CudxWnS7ukw
CwHSb1NOi4c
CwImmV7q1MY
Cwbtn_h6TP8
CwmUMySSNQc
CxTFgimfNfA
CxgVq6eovRU
Cy3HWnwMLyI
CyFsI_EYFQQ
CyiPyjYX6AE
CzHZNJEV-3o
CzMNiypg1I8
Czbi1u-gwUU
D-PxXM2I5gY
D-p9s8y2z_U
D0L-M4trkpw
D2w3qHmJrdU
D3FyfFIKLVc
D3f5VIJYR7M
D3ht_xXl5S0
D3q_stCeCA8
D4GVgBX1eYc
D4ccFYk3bhU
D644BWAUOXo
D6xrH93lnoc
D712KM8PE3I
D7Cvisf3jf8
D7pjR9cQChM
D7z2Q-hH25s
D8-x1T8M4gk
D8zK7PHIkgA
D9893od03Sc
D9TuP8PKD6M
DA3fNvbZoBM
DA8lw6Mq0DY
DAPGvg8qOAU
DAX9uKYlDvw
DB38NRSHw9A
DC0C-KO9EJk
DCFrCX4HPO8
DDZlMjb-0i4
DE6bdmnmPtg
DEnayQZiPGc
DG5d4megH8g
DGON0D8E17Q
DGbMEkQerYs
DGlP6oTqe5Y
DHDn79ee98Q
DHo1z0_ZUNA
DJHEBMNPc-A
DKBbLySEGic
DKflAAykh6A
DLDZrtwyY_Y
DLJGT99uEh4
DMwvY--3XD0
DNYDyXn6qso
DO8Db_i7IVc
DOb8htND5_o
DP2vmsftZHY
DP72GDe6AAs
DPW5M1FYxu4
DQ_gcdLhAsY
DQrdOgRb-oA
DQzg3cZeYSw
DRU-IFx-7yQ
DTkKGYCRMlc
DTnCrtCro44
DU3Vlaa8rU0
DU5pD63Pv30
DUNOn71oGCw
DUmXpq-Cksg
DV5mynb77JM
DVuWm53IlVw
DW3z-ByrfWY
DW7Qe-50X_A
DX4_QlYLq30
DXd83S6NHLg
DXeiJpZXVAI
DY7ko7iQCug
DYp8940tHso
DYpjbiyPUho
DZ3EShJzOOc
D_QEW1Lnl2Q
D_usHKfOXCw
DaeJYtWgcoI
DaiVfxATCEE
Db0c4aAwUbY
DdxW_JziHTA
DewyiIpkzi8
Dg8BLvkzdr0
DgVpK6r7JT4
DgqPgNqW2hE
DhnNZn8JjEQ
Di7Rs4QmYKI
Dk7ziLlNfeM
DkZ1Or_D-g8
DlU56FGTVEk
DlVgHV1UobM
DpAfyjjQJyk
DpS_TigOHWo
Dr4Ijx7Q9jk
DroAzooK4yw
DsAp3b1poeA
Du3Q6NdsSso
DueNcVFHI0k
Dv21JlCT4_k
DvN5813H7Bo
DvOA0K-DIFM
Dx58W7opw5A
DyPmDDN8m78
DysXetu2I0E
E16FgDFQI_w
E29kpquu8W4
E2gstPe3Im4
E2v025Ilsqo
E3F9bzeCgTQ
E3iPiZlT6f8
E4To9BC2jx8
E4a3s8NRqUM
E6B81rrUlnE
E6JR3htwgyE
E7D42u8a5gc
E7q_QwLYI8U
EBpa2CADNJA
EC4GbkL3XvI
ECP7EJka6N8
EC_JNTVDrok
EFlGHxV5WsY
EGIeykrN4eg
EHm7vLZewS0
EIzBD62ja8E
EJMUwh1vY2s
EKZvq0dUk50
EKt_KEbqQfQ
EL2DtgPD4J4
ELhxZhWsGM8
EN_FOFkxAEw
EO_tdzbN75Q
EOaQnfDjVyo
EPjfqS5NvCY
EPvnkbo5wrI
EQHrQIaQNv8
EQSv7DIJJtw
EQVZLtlDkHw
ER1fTL9-pQw
ESFANzZTdYM
ESbtW0CUmp4
ET6HVZ8muVE
ET7yQfaiF_8
ETl0hYRlVmg
EUNTykrvpok
EUmfsCvmkgo
EXRKJRL0TDU
EY8boPZ1hPs
EZAwPnGOJPE
EZJzzWEDtQo
EZb1wQsg6CU
EZi80fUpBfE
E_34BmDVnOI
E_IG2ubeTME
E_kgtQ93Q1s
EaGhKzpkNso
EayN_Jj0740
EcEDVL6vn_w
Ee6MP1bIRUA
EfUUgsioXyU
EgwGYmAH0BA
EhFWLbNBOxc
Ehks1uuwR3s
EieK70X8lnw
EijTwCm-pRM
EkMgPJfSL04
EkmHGd0U8yE
Ekoh6TJpA88
EmSZKb0LdVM
Emc18GpAeRY
EnJ8zV3vaJA
EnewI6fNhVA
EoZH1gyRlr4
Eop_sG9FVgA
EptdhC17avY
EremEyLrdUY
Es9FNjZ-SHI
EsHXnkZ_W2c
EsssGCL-Axw
EtmKuWPpjG0
Euu6zlJQSD0
EwDiNj_5PEg
EwoCbcSXlSM
Ex0ukzO5z9M
Ex18Xwznj60
ExghbCGRBx0
EyO5vB4eqo0
EzP7PB2x670
Ezodz2aZnzQ
F1X7egd8Us0
F1uXNtotVsg
F2ekiX14ID4
F3uf_RleI3E
F5e-SEICJP4
F5xnAYHuGlo
F5zDEHggiMg
F7JllgnefSI
F7Y1T-rQ2JU
F8MCOgWhgvY
F8wGRd9332s
F9LJIyqQFe8
FAhfuP_xh5Y
FBXBDe3OU5s
FCvs17jk10A
FCzMqo8kh1o
FDO5BekX478
FDYIdBZUl2Y
FE2kALvHjEU
FENJIDecy5s
FEuXIeWoCQQ
FF0VaBxb27w
FFQVVwFjy7s
FGZ0sLt4dXA
FGoDfNZezh0
FHC7gN3NnX8
FI_Dl0Cmd38
FKBryvLMTY4
FKChZXXhufE
FLFmcZiRuzU
FLJPxFCPeDY
FMjUGdTn7lU
FMr1WHtXJbc
FNnKFUkFJ5M
FRZzUh9hcTo
FSQ3E4XbpPU
FTdcanPJw6E
FXQxobF8FWw
FXVu-YwjhxM
FY8bZRm_QiE
FYFapDVOFHg
FYux89o7Hhk
FYwDTJtEzhk
Fa1KdG8niq0
Fa1c4qfBqzE
FaD9qs2ACpI
FaRrq7cYu84
FbV2Lgt3H1Q
FcwWl5JBnoU
FdPaa6WR2F8
FdvGsAq99r0
Fg8yzA7zifw
FgUSmmTEE3M
Fh53p0Tnnxg
FiKY19GK6-8
FkpJaXzgMBQ
Fm6ss1PBC1Q
FmytwY7SAfU
FoFMRXlNJ6Y
FsCQmTluSDw
FscE_AHEmFk
Fsm-xDmyFKg
FsnRM2irjvI
Fsv_syCvzsc
FteW_2gNtD4
FtskdD6Py7Y
Fv9swdLA-lo
FvIKye4-iGc
FvQgVl5IBHw
FvdUm5j_oA0
FxeA9blzUD8
FxpV97ILuSo
Fy51z2RwH3E
Fz-ZNyVKWro
FzG8ZQAhKrE
G-5UYGccne4
G-7BfzMgZL8
G13NEVAm6-o
G18Nt0ZeEJQ
G22YfD5xxMU
G2JDDwIuNrQ
G2uCAwYS6w0
G4fTKotMoWI
G6A9NKeK8ko
G6ihF82lvEA
G7pD1K3jYg4
G8iTuTi8ywU
G9gsCU85c8k
GAoGADilmV8
GAohd8KvONo
GBLKj2d0iC4
GBMlv6j0WJ8
GBsate-JQAo
GF8QWSW0UbY
GFGwPa9d9XQ
GFJNgqcX7u0
GFT6TeQx_0U
GFbSHWBjuuQ
GG6XkHATIyw
GHQlBD-6rkA
GHyUAl9Yaos
GJYhDjThTHM
GLIXnXZEOxY
GMFWnMRtfNI
GMtb7U-8IYM
GNjsxLdSwHI
GNu_hiHGEp0
GOTy3yhCylw
GOujNXEtDmg
GPSqrciDLog
GPwzpw_47Dg
GQPOpFX20Gw
GQbUpJFArKI
GQbytKkt0tg
GQz_u0Vc8Os
GRdzFvQezUE
GSAMEMX_oAg
GT9g4uqK9G8
GU782ddSBvk
GUwBLItoJXk
GW6pti04qIo
GWcMqKYOJR4
GWcTd6XrQj4
GWxiiDBK5s8
GX-QhoihLeI
GXLeUXSVFYU
GYCfrx0ruz4
GYx6HNQbP1w
G_iJif-fC6E
GaNjXwElAUE
GbjtSTTEFK4
Gc8xf7CJiFY
GcOOmVSM8Uw
GcYeBWujhjw
GcbCOmNiVm8
GcnLApOeCh0
GcqCHmHXEjo
Gde1fn1Y1uM
GglIHiqClGM
Gh4KBdcCU64
GiTmjE7az74
GinP_ZRgAoY
Gj6etuzTWlQ
GjE_iD2BbFg
GkB_BkyVyPs
GmGWvBNO8JI
GopccU3Am1w
Gow0TlxIx7U
GrbrWNohr6Y
Grtmre_r9yI
GuHDy--gWiM
GuJdy864xWM
GuQFhmGqdog
GuYRF0no7hw
Guu30szkA-0
Gw0KmZ3kbjs
GwTXkfuc5v8
GwxSvUoYSZg
GxeDIuOzxmU
GyxH8ep_Vx8
Gz8RlHf3Czs
GzRvq0gJbj0
H-bTMbePj0A
H18aK9HhNSM
H4rdJlSSt5Y
H4tyvJJzSDk
H6Y_7Ax34-g
H6qzijVEqZQ
H6rZwBc6aNM
HAEoz3VbaP8
HAHn_zB47ig
HCCGZh-TxK0
HDSV7Pzq8CA
HEclHruM37s
HEhogaw0vUg
HEk4fx_XNNE
HFH9tcIK_PM
HFVM5pVTwkM
HH3ryAEP308
HHTgjmgTV6c
HHZGjS4g-w4
HIDXdH6R6T8
HJuR9WJ1iRY
HLXGkzzQHUU
HLsRePLObfI
HLz3N5nG8fQ
HMJe5jS0Yt4
HMQfp_qtF-M
HNf9eHqDT1A
HOIIp5NyFx0
HOkuea1wExA
HQ9HlWProm0
HQb2jhmw1BE
HRVVqstIabc
HRxTN4TH-80
HS_ikHx4LIQ
HSrzR9hhEe8
HTQySJM4Jhg
HU7oqkJeItQ
HVA9-fjtv6U
HVsXJDR1_Lw
HWZdPxRzCWs
HWe3TSUcvHc
HXG8DnTpyPc
HXwX9f9ugZ4
HY1KdNS19CM
HYjSrwSm0T4
H_2ZPxy80Eo
H_5wh4aMQe0
H_He9_zHk8I
Ha-FS_CHmGw
HbX1ZIuD0ac
Hc_UM8l_sTg
HdRPdh-cSTw
Hdko95EitpI
He63KV_9Pwg
HfBh3lZZi8Q
HflVPAOVL7U
HfzEa06vDLg
Hg4f2xt3oKA
HhQTvaZtURY
Hij_QxDkIJI
HkCYA4ax4jI
HkXSX7Kdhms
Hl8OrRlMwi4
Hn46VuvS88Y
Hnk45Z0EAxg
Hob0LAu8afQ
Hp-lgcs4VXY
HpkPTa1fQDE
HplcVmJhuIc
HrPnGYGrvm0
Hrgl_1rGGU4
HrwpdRe7meM
HrxfNVYirCo
HtCkwxfAmzw
HtRjRuKnvjI
HtSznF9_784
HtXtfCR-MUg
HuVgc2jf0Ec
HvOSaS8sXQM
Hvs6Xwc6-gc
HvuHSr_yncE
HwGK5RvNOFI
Hxf1seOpijE
HyJ2YaNrA3U
HyXMWRU9Owc
HyyHwIK9SSI
HzXWXYxXyYA
I-C14nCneBs
I-XYm2Ck2r8
I-Z3gB6pfIA
I-xPuRe9vF0
I06TOd9pXng
I0q3IGmTkRo
I0skZ6yT36E
I11AcD1sGes
I1fcUe9MoMw
I1wakVlpP6M
I2yA-F-_A2E
I368EWBLIs4
I3qbB4Kq3Y0
I4Jp0kB2Ns0
I4Rhe1XViYg
I5CBPhpimtg
I5KHYgtrVBw
I5mESbabhZY
I6eU2qRjJ7Y
I6wri5XQQbU
IAeYandwJqc
IC_Wpalzzm8
ICcASgMtIJ8
ICtri0ElFZc
ID4AoAfHMVk
IDHHAwqDpzU
IEVVHo9nr7g
IF-77lLlMzE
IFimpFwvbz8
IFumVgqOVaM
IGAzIIZRczw
IGyDtKAU1u8
II1oyaWPiD0
IIK0EuHyb6w
IISJXKl1Ih4
IJcLW4arT6s
IJw2o_Yg00Q
IKfmu7tAyUE
IKq2OF8jq1c
IL1n6jzABVw
ILE12hEW5Ck
IMnh-TIyFuE
IN71kMOAk_k
INBx8CrIWcg
IO5QJRyoqO8
IOEwc-N2rX4
IOzWDVGWRng
IPO79GrOYjg
IQbzpgmi4Ec
IQxr3xwAbKk
IS5V2yjPp3k
IT5hcf0KhYE
ITYv4126yhk
ITg9o6Gwsbo
ITq14s8h92I
IUD_ZYOh2MM
IUjzBX2Qm4k
IWbe-NSK6Ic
IYiHhVWrh0w
IYtQfDsFVfA
IYumekd1VMg
I_a9OhfTcIc
I_kUf7vgVNM
I_wT76iYBdQ
IbD0zpcimgM
IbJh1xeBFcI
IbaQjxGDT-o
IcPbxJRbe5g
Ie5FO_BetOE
IePfzUzlDng
Ieef9l-gWN8
Ife1WaGirdQ
IhNPDueFVSo
IiBgER0W8iA
IiJciyiQBDw
IizUHzmcPGA
Ij1-640aafg
IjP5kKfgBiI
Ikdb_jA9ehU
IlUcHzBzZvg
Ime3FHuQG4k
In44gO8Ej90
InKK8z21UYo
InQs7K9MqaI
InfQUMh935c
Inuq5W98ktA
Ip6FptuXHyk
IpYw4nKPx5o
Ipow026xzVw
IqGB4nQIAcQ
IqV9wp0IeAo
Ir82jewhXCo
IrIKXhYuwuU
ItLWKIhe58c
ItSLkF6O3Mk
ItstzW7xuDU
IwdjPDw6o5I
IwobTmzjOiQ
IwqD859w2_E
IyJ3a5uuCOs
Iz611KubW70
J0RzvGnQxD0
J0lA7ZDfPLE
J0t4VMnXDNM
J1-Qvl7u2TI
J1gZRam89EE
J1nIXpnMe1U
J2BDMndrvhA
J2R8Ab25reU
J48F0e0guSQ
J4DrdTy52kg
J4tjy-0CNm4
J7d3nuS9wqg
J7fVdG7Zd3A
J7jVR6y6REA
J8lCxfaiHeo
J8pkQfYlJA4
J9PJI1UwIQ4
J9ZlahUawkg
JC41M7RPSec
JCfZpSEH77Y
JCsuOzlwJqA
JD9LYReBGXU
JDBu-3VCyWc
JDWPJ1AiDKc
JDrnf3vldLw
JEJLTct-014
JFGNmPzPXeA
JFJuEOZx1K4
JHkcCXF5vII
JHvLuYk6TfI
JI26wmUPcrM
JIoA1KsfioQ
JKbUVdMJAO8
JKihzveDE5g
JL4Z2_5Q7sU
JL9Xxzf9e2A
JLYb7DwCaQU
JNW76fDWkA0
JN_VJCJhC4Y
JNw0A8pRnsQ
JNwt0afB1fQ
JOhK7oq9KtU
JOkuwbhMxbQ
JP637fg_ZC0
JPVRBbdykSw
JPZlyvPNZj4
JR-1k8GHYAw
JRfU_hF1wdM
JRmfjBDKCpE
JSLqi_TsMzs
JSdALuTneBM
JSqyTVjYY6k
JU4CZ-GApu4
JU6GUqRqbtI
JUrYWttZJBM
JUwu4xOs8K4
JV_IOR3DqiM
JWNWCKdfpzM
JXS5eW7g4HY
JYYfw3id3ek
JZWJlUdYpCU
JZnOGRCBW0I
JZyw6YUsGzo
J_Raltj-6dk
JaGUY5ULTok
JbWaSZPOh18
Jcd63Ev7JXA
Jdy08IPLKdw
JiBAkAwK0GM
Jj9orXFko0Y
JjF1G5wgcvY
Jjr0_CbcYdg
Jk2mvFrdZTU
JlzlNpttvVM
JmbrGzgxrJ4
JnfMv9ti9Sw
JoBRbtAnbVM
JorNR3IXDyc
JpMHnsdsCiY
Jq2w30NYstQ
JqmOqYtQqB8
JrOV2Xz2djA
JsHTGDW5dgw
Js_3Aa214xY
Jsk3ZUGvP-o
JtLNRHVGQuw
Jvj2WqgVy78
JvqCsVj0I4k
JvvnL7UnCGA
JxwRvkjNJQ0
JzJCn-puzS4
JzRb1OVpat0
K-0qmhvJyzE
K-zkbbliQcI
K0x_DxNxtbk
K1PzpuR6CqY
K2h6UiZSoZ4
K55v5p5DEPE
K5ilD6nEJ-g
K63Z_abB314
K6DSH7MSeOA
K6KbEnGnymk
K8On7nUJuP8
K9zE9x2ccJk
KB4e9v_5uTE
KB79k456DhI
KChjW89XOF0
KCs_VPmsnKo
KCytKo5LzCc
KDuusOmEMHg
KDzy3ZL626U
KE-TVQhdCbs
KEp2NhraIZI
KFB1raoIhoU
KHjEIheD-Cg
KJHqQ5aKu8U
KKgYvcfrxj4
KLFoZA8btu4
KMQmM12G9Z4
KN9vuaQvld0
KOb-tRHYK68
KPG9s_s8siA
KPnIFb7T7VM
KPymcVenomk
KRKX_UtYV9c
KSFND-AdqZs
KSye2ifWZ_Y
KTqf_FXZygM
KU2bOYiBX28
KUE_I30--AY
KV4noVyGHa4
KV99GJg0tvA
KWpsFxRTGkI
KXRngbBe0bg
KXuB62SMFvA
KY5TY6ovKQg
KZ8gBHLNmH0
K_G_k1WTdoc
K_ZuxYxxT60
KabP9iNokps
KauCO8zH5bw
KbDLu4VozGg
Kd7aHdOwh0I
KdNhYvN4Xoo
KduU8kmRsLg
KeSbjmMeyrY
KfAq7kxHjPk
KfLxBjl-aBU
KgMD2_Yhw7Y
KhXcEu7r6m8
Ki7Bxz1CThI
KiFQFxJphjI
KikHXfUanJk
KjJj5-HvSvQ
KjMRf4egAyA
KkW6ZkmAlEw
Kkqbi6c_40k
KlD1u-EDx_g
KlVO3gu-j70
KmBaE7ozWow
KnZeEWVHaLc
KoAGZ_dB8MM
KoP8T0QqzeM
Kojo5khAAS0
KpRHRh9PaRU
KpwdlYIdtfs
Kqo7am5oq0U
KqtlecvEOGw
KrK8Giu9ZUc
KrmG43H1u70
KsFCPOKAH60
Kt2MwCHV3Ko
KubrAnJ0o0o
KurvLmoKCog
KxUcwQ5BEBk
KxVbdGPAfjE
KxZ0yDfyaJw
KxjiEi2Eywk
KyRJP_fDrbk
KyjeM7J-Pz4
KzvdKLdBw3s
KzydTOkZty8
KzzKguqINa8
L0-l1LIa22g
L0oun9F67tg
L1fxVVl_1bM
L1s-oPHsOac
L1s7KZgWXGc
L2-EGNKzUAQ
L3ZtE3rD2nc
L3d4ajg0bhk
L47F51OmZXo
L5CgdTtGv8o
L5UDz2PJ9sk
L5Uu_0xEZg4
L8zjEoQFws8
L9GXrmmlYhE
L9j9fCHHPeg
L9xj_v65UhU
LAHWV6fZwUk
LAeWwMC2EaI
LAxCq-s84F8
LB0u0PrlDHU
LByEH80c6Eg
LCMQXFKMLIM
LClTjcyNJSI
LCzldLY3E4g
LEG7xkYOsWA
LF-5BAUGvWI
LFW4yPH1z3M
LFYRuK8YstI
LGW99kSaf6M
LGi38MqlPFA
LGrAwQnZ6Bs
LH3mAtCou6g
LHTA8VZGoMs
LJsYl38zPOQ
LK6zk03lPlM
LKUYtvUHn0Y
LKVSEQCLCLo
LKurVRvkmKc
LLDHkHHKYkI
LM2C1eIUX9M
LMGpKPavV4Q
LNFAeJ06KVs
LP1yZRsRllQ
LPA8RDzl-Ss
LQYd-dsz62M
LR3U4b_fVBc
LRUdmYcXFuM
LRfVQsnaVQE
LRz6Y6oltQ4
LSaLPObrnZw
LTDle_h2YD8
LTG-uVV_6q0
LUtBqNS27AQ
LWHUat2fo9w
LWIHz5kao3g
LWLLN9INTII
LYO7_GxyaZc
L_KkjB0Wt_Q
L_fWnna7Np0
L_ghM-NrH58
L_nC2BvhRdQ
LaaC_q3QDUE
LadgIxZu8Oc
LaeRCg-NdeY
LaoUSBEVHVQ
LbPRGDwlfqs
LbQ4zHxhoSI
Lc6OfmzV7Pk
Ld5G00HlbQs
Le4aGNS8e0c
LfbGHMumxIQ
LfvdxSBCtFE
Lg1HG6D_0Qk
LgKOnHwaCmg
Lhw0H3P4zUE
LjihfG0fit0
LjxhswBV1UA
LktT1OQrPdo
LkzyG2F8mTM
LnL3KFKGqHE
LnczSOwV9Ds
LoTRWc9WK9Q
LpcvkXs49Zo
Lq0LMMZfHCU
LqP4F4a-HOc
Lrhy2auO6hY
Ls6qMcgpdlM
LvpImXJMryI
LwmwCpAVPWU
Lwvf17xUxhE
LyTwxJiSt7A
LybSS4amIS0
LzSWdj4izHM
M-RX7LqL50A
M0ygCD6WyXw
M1ds6tRFxhs
M27mIdPCZEY
M2iOUwFHv9Q
M4SIUPA05yc
M4rXhyyvERM
M4yst1q1nlU
M5sptjrboqA
M61BBJpvvx8
M7GSBDc6s_w
M7fsgHyiicM
M9MgZXkYRBs
M9mc3HYL_GM
MC0Aeu7RLSI
MEew7OQ17HY
MFxMPOAbUPA
MHHshnnqyco
MHgPpImV7b4
MHkfPjW0aRg
MIexFfOsuJs
MJtDDmS6xSY
MKXeCiPtZwo
MKikHxKeodA
MM0seezR2F4
MMGAKKhqxKg
MNd8CRa48Uk
MNlzpCwdh4g
MP7KPlqoQW0
MPe6ztPtF0Y
MPxwPOOIskc
MQ1Q7xydJPU
MQrOnSzVlJg
MROotmz8a-U
MS8VU468rKk
MSHbLrVlrQc
MSqQKf3O3vY
MV4tgzc9X6s
MVAXHT67Na4
MVBQrBAXgw4
MVYSWTF11Nc
MVq9PYtypy0
MWS-Uxf1MRw
MX0wS7MX3Zo
MXdVnDVjSL8
MY0PsDE3xHs
MYtq46rNsCA
MZhaDGgULtc
M_s-49rNCdw
MdYXznF3Eac
MdsRmMxkF4k
MeA8CSKAuvw
Mf6Ql55o7Es
MfKSVJIcDK0
MfX7Q0ucts8
Mhvgz5AjV3U
MiWskwqOMrg
MipnqUXgpOA
Mir959i7F2M
MkPhe7TLLZ0
MkTQQ0m8Ys8
MlnK2sa7mm4
MmqRlHntd0Q
Mnk6590abfY
MocXmVbat3s
Mp1MHSeHa0o
MpS2SSIhe2g
MpWGx5odhh8
MpjN21Z93JY
MrMXYO2fzJ4
MsEoUVWS59M
MsjeOXuUYG4
MtVLmOvQopM
Mv90uA0tmgc
MvnC1TfNiPY
MwE7REVj8JQ
MwK9HYjeeN0
Mwy5Y0S5jfM
MyH8zQw9csc
MyjxrBI9k4o
MzUgHy7SyS8
N-dzfI3L5ic
N0i99VyZCg8
N0q5vPAsHLI
N1Mxns_JJTk
N4eMppEnPE0
N4tTZn8WlDM
N5BAnG2zoUY
N6YT8_jlt0E
N8Fg3L1Cc5E
NBnz0xV9nb4
NC5tIv4-8fg
NDJEKij2qOg
NHA1l_Czm38
NIcsJ8sEd0M
NJGo2fmUAII
NJ_Ha89QjiI
NKDBwJIwWoU
NLQts9t7d8k
NNeEzTVATHg
NP5iO_HB-f0
NPXZIqxKvXA
NQXQsVawPhU
NRWlHRvaDcQ
NSS9_2FFVeo
NSyqj1DXZKg
NUTaOnEhvzE
NVo-stvk_QE
NWL-P08eM-U
NX3KJ-tVdMI
NXuB3ZEpM5U
NZ2kFIaW05k
NZYDLDIyZr8
NZn4-gP2GiI
N_41Y2vH6eA
N_LKZjw9DLk
N_Wx35sNqdM
NcsYdCbKgcc
NdiSW-p2I0c
NeSIT6sqLRA
Nep_3Y81E_w
NgniX3tg_Mo
NhVkzcCL0SA
NjoKxRwQxCE
NlCfScKw_Mk
NlUf1ppoSG4
Nlg8AbWRV_c
NmMJgUo19Gk
Nms2A0wi0vU
NmwmOY6iBFg
NnYfF7E12dk
NnmJ1UHWlas
No4Bi6mG1J8
NoB_4XaZYVs
Npbs_4DZgEQ
NqDxpJ2uR_8
Nqb7nw58q08
Ns-iXXKmzzU
NsR60ehkHGA
NsYVaRI6rXg
Nt0U-CXK6O0
NtM3gudMBCQ
NuN-ug3dIkw
Nuks8XFdGMk
NwA9JSlK_lM
NwUJe24OxC8
NwfEO8cjSK0
NxdQtpceXaI
NxpnW_IdkSY
Ny4wnDN4K1U
Nymjfq2kXnI
Nz4iLzJBTBo
NzVg-cFQJbE
NzvkMWY4EjA
O0sDg-yLvlE
O0y-m0pCi5E
O1EmHJyz5ds
O1RmrE_HfpE
O25IKwo2HkE
O28kY0aN8VI
O2HttJtcec4
O3Cvn4yXrao
O40E8bpmONQ
O5IulN0n6d0
O66lIRbF4Gw
O6QyYC7Tt2A
O6xMQnKJROc
O73wigUotGo
O84YjlJ_Qw4
O8EMm4QJjeo
O8rHjrG3HM4
O9Ag-dE-yfQ
O9_avJFKIQk
OA63sRxGk6o
OAl2EjbdQG8
OB7GyVqufwQ
OBJM1TqPvu4
ODOrls3MuZI
ODRAYQE9GXs
OECgm1obaFs
OEIj1UX5ZRg
OEjgIDubFbg
OEpMpYMjO9Y
OEuBITrf-kE
OFJG5Wo_knI
OFP5MYVDa4g
OH2SQhJqZDg
OH8urnIthoQ
OHZZuO2vY50
OHrrS6AKW_c
OI7S7vaBT4I
OII3VJoE0WA
OJuVsBojdvo
OKZF0oG1E14
OKquGBKOgME
OLy3C8YpMsY
OM04OFjNPGw"""
def normalize_ytid(x): return str(x).strip().replace('\\_','_')
seed_blocked={normalize_ytid(x) for x in BLOCKED_SEED_TEXT.splitlines() if normalize_ytid(x)}
existing_blocked=set()
if USER_BLOCKLIST_PATH.exists():
    existing_blocked={normalize_ytid(x) for x in USER_BLOCKLIST_PATH.read_text(encoding='utf-8').splitlines() if normalize_ytid(x)}
USER_BLOCKED_IDS=seed_blocked|existing_blocked
USER_BLOCKLIST_PATH.write_text('\n'.join(sorted(USER_BLOCKED_IDS))+'\n',encoding='utf-8')

try:
    failed_cache=json.loads(FAILED_CACHE_PATH.read_text(encoding='utf-8')) if FAILED_CACHE_PATH.exists() else {}
except Exception:
    failed_cache={}
PERMANENT_FAILED={normalize_ytid(k) for k,v in failed_cache.items() if isinstance(v,dict) and v.get('permanent')}
REPEATED_TRANSIENT={normalize_ytid(k) for k,v in failed_cache.items() if isinstance(v,dict) and int(v.get('count',0))>=2}
SKIP_IDS=USER_BLOCKED_IDS|PERMANENT_FAILED|REPEATED_TRANSIENT
log(f'Pre-filter: user blocked={len(USER_BLOCKED_IDS):,}, cached permanent={len(PERMANENT_FAILED):,}, repeated transient={len(REPEATED_TRANSIENT):,}.', 'FILTER')

PERMANENT_PATTERNS=(
    'private video','video unavailable','this video is unavailable','this video is not available',
    'please sign in','sign in to confirm your age','members-only','copyright','removed by the uploader'
)

def find_audio_file(ytid):
    ytid=normalize_ytid(ytid)
    for ext in ('.wav','.mp3','.flac','.m4a','.ogg','.webm'):
        p = MUSICCAPS_AUDIO/f'{ytid}{ext}'
        if p.exists() and p.stat().st_size > 2048:
            return p
    return None


def classify_failure(stderr):
    s=(stderr or '').lower()
    permanent=any(p in s for p in PERMANENT_PATTERNS)
    if 'ffmpeg exited with code' in s: reason='ffmpeg'
    elif permanent: reason='unavailable/auth/private'
    elif 'timed out' in s or 'timeout' in s: reason='timeout'
    else: reason='other'
    return permanent,reason


def record_failure(ytid,stderr):
    ytid=normalize_ytid(ytid); permanent,reason=classify_failure(stderr)
    with failure_cache_lock:
        old=failed_cache.get(ytid,{}) if isinstance(failed_cache.get(ytid,{}),dict) else {}
        failed_cache[ytid]={'count':int(old.get('count',0))+1,'permanent':bool(permanent or old.get('permanent',False)),
                            'reason':reason,'last_error':(stderr or '')[-600:],'updated':datetime.now().isoformat(timespec='seconds')}
        # Persist immediately so a Colab termination cannot erase the failure knowledge.
        write_json(failed_cache, FAILED_CACHE_PATH)


def download_one_musiccaps(row):
    ytid=normalize_ytid(row.ytid)
    if ytid in SKIP_IDS:
        return {'ytid':ytid,'status':'blocked_skip'}
    existing=find_audio_file(ytid)
    if existing is not None:
        return {'ytid':ytid,'status':'present','path':str(existing)}
    start=float(getattr(row,'start_s',0.0) if pd.notna(getattr(row,'start_s',0.0)) else 0.0)
    end=float(getattr(row,'end_s',start+10.0) if pd.notna(getattr(row,'end_s',start+10.0)) else start+10.0)
    if end<=start: end=start+10.0
    outtmpl=str(MUSICCAPS_AUDIO/f'{ytid}.%(ext)s')
    cmd=[
        'yt-dlp','--quiet','--no-warnings','--no-playlist','--no-part',
        '--retries',str(YTDLP_RETRIES),'--fragment-retries','0','--extractor-retries','0',
        '--socket-timeout',str(YTDLP_SOCKET_TIMEOUT),'--concurrent-fragments','4',
        '--abort-on-unavailable-fragment',
        '-f','bestaudio','--download-sections',f'*{start}-{end}','--force-keyframes-at-cuts',
        '-x','--audio-format','wav','--audio-quality','0','-o',outtmpl,
        f'https://www.youtube.com/watch?v={ytid}'
    ]
    try:
        cp=subprocess.run(cmd,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE,text=True,timeout=YTDLP_TIMEOUT_SECONDS)
        if cp.returncode!=0:
            record_failure(ytid,cp.stderr)
            return {'ytid':ytid,'status':'failed','error':cp.stderr[-500:]}
        p=find_audio_file(ytid)
        if p is None:
            record_failure(ytid,'yt-dlp returned 0 but no usable audio file was produced')
            return {'ytid':ytid,'status':'failed','error':'no output file'}
        return {'ytid':ytid,'status':'downloaded','path':str(p)}
    except subprocess.TimeoutExpired:
        record_failure(ytid,f'timeout after {YTDLP_TIMEOUT_SECONDS} seconds')
        return {'ytid':ytid,'status':'failed','error':'timeout'}
    except Exception as e:
        record_failure(ytid,repr(e))
        return {'ytid':ytid,'status':'failed','error':repr(e)}

# Count Drive-persisted successes first. If 766 are already there, this cell performs ZERO YouTube calls.
usable_audio={normalize_ytid(r.ytid):find_audio_file(r.ytid) for r in df_mc_raw.itertuples(index=False)}
usable_audio={k:v for k,v in usable_audio.items() if v is not None}
start_usable=len(usable_audio)
needed=max(0,MUSICCAPS_TARGET_USABLE-start_usable)
log(f'Usable cached MusicCaps clips in Drive: {start_usable:,}. Need {needed:,} more to reach target {MUSICCAPS_TARGET_USABLE}.', 'DOWNLOAD')

# Candidate order is deterministic. Known blocked/cached-failed IDs are removed BEFORE any process is launched.
rows=[r for r in df_mc_raw.itertuples(index=False)
      if normalize_ytid(r.ytid) not in usable_audio and normalize_ytid(r.ytid) not in SKIP_IDS]
log(f'Candidate rows after pre-filter: {len(rows):,} (filtered out {len(df_mc_raw)-len(rows)-start_usable:,} known rejects + {start_usable:,} cached successes).', 'FILTER')

results_dl=[]
if needed>0:
    # Rolling executor: do not submit thousands of doomed downloads up front. Stop scheduling as soon as target is reached.
    row_iter=iter(rows); inflight={}; successes=start_usable; attempted=0
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
        for _ in range(min(DOWNLOAD_WORKERS,len(rows))):
            try:
                r=next(row_iter); f=ex.submit(download_one_musiccaps,r); inflight[f]=r
            except StopIteration: break
        pbar=tqdm(total=needed,desc='MusicCaps usable clips needed')
        while inflight and successes<MUSICCAPS_TARGET_USABLE:
            done,_=wait(inflight,return_when=FIRST_COMPLETED)
            for f in done:
                inflight.pop(f,None); attempted+=1
                res=f.result(); results_dl.append(res)
                if res['status'] in ('downloaded','present'):
                    successes+=1; pbar.update(1)
                if attempted%50==0:
                    failures=sum(x['status']=='failed' for x in results_dl)
                    skips=sum(x['status']=='blocked_skip' for x in results_dl)
                    log(f'Attempted={attempted:,} new candidates; usable total={successes:,}; new failures={failures:,}; blocked skips={skips:,}.', 'DOWNLOAD')
                if successes<MUSICCAPS_TARGET_USABLE:
                    try:
                        r=next(row_iter); nf=ex.submit(download_one_musiccaps,r); inflight[nf]=r
                    except StopIteration: pass
        pbar.close()
        for f in inflight: f.cancel()
else:
    log('Target already satisfied from Drive cache; skipping MusicCaps download phase completely.', 'DOWNLOAD')

write_json(results_dl, MUSICCAPS_AUDIO/'download_log_latest.json')
usable_audio={normalize_ytid(r.ytid):find_audio_file(r.ytid) for r in df_mc_raw.itertuples(index=False)}
usable_audio={k:v for k,v in usable_audio.items() if v is not None}
# Deterministic cap to exactly 766 if more clips become available later, preserving reproducible splits.
if len(usable_audio)>MUSICCAPS_TARGET_USABLE:
    keep=set(sorted(usable_audio)[:MUSICCAPS_TARGET_USABLE]); usable_audio={k:v for k,v in usable_audio.items() if k in keep}
log(f'FINAL usable MusicCaps paired-audio pool: {len(usable_audio):,}/{MUSICCAPS_TARGET_USABLE} target.', 'DATA')
print('Failure-cache file:',FAILED_CACHE_PATH)
print('User blocklist file:',USER_BLOCKLIST_PATH)
print('Both are on Drive and survive runtime termination.')


In [ ]:
# Cell 4 — Inline audio feature extraction + graph construction.
# High-information node features: mean AND std of log-mel/chroma/MFCC + spectral summaries.
class AudioFeatureExtractor:
    def __init__(self, sr=AUDIO_SR, n_mels=N_MELS, n_mfcc=N_MFCC, hop_length=512):
        self.sr=sr; self.n_mels=n_mels; self.n_mfcc=n_mfcc; self.hop_length=hop_length
        # 2*(mel + chroma + mfcc) + 2*(rms, centroid, rolloff, zcr)
        self.feature_dim = 2*(n_mels + 12 + n_mfcc) + 8

    @staticmethod
    def _mean_std(a):
        return np.concatenate([a.mean(axis=1), a.std(axis=1)])

    def one_segment(self, y):
        mel = librosa.feature.melspectrogram(y=y, sr=self.sr, n_mels=self.n_mels,
                                             hop_length=self.hop_length, power=2.0)
        logmel = librosa.power_to_db(mel + 1e-10, ref=np.max)
        chroma = librosa.feature.chroma_stft(y=y, sr=self.sr, hop_length=self.hop_length)
        mfcc = librosa.feature.mfcc(y=y, sr=self.sr, n_mfcc=self.n_mfcc, hop_length=self.hop_length)
        scalars = np.array([
            librosa.feature.rms(y=y).mean(), librosa.feature.rms(y=y).std(),
            librosa.feature.spectral_centroid(y=y, sr=self.sr).mean(), librosa.feature.spectral_centroid(y=y, sr=self.sr).std(),
            librosa.feature.spectral_rolloff(y=y, sr=self.sr).mean(), librosa.feature.spectral_rolloff(y=y, sr=self.sr).std(),
            librosa.feature.zero_crossing_rate(y).mean(), librosa.feature.zero_crossing_rate(y).std(),
        ], dtype=np.float32)
        return np.concatenate([self._mean_std(logmel), self._mean_std(chroma), self._mean_std(mfcc), scalars]).astype(np.float32)

    def extract_segments(self, y, window_seconds=SEGMENT_SECONDS, hop_seconds=SEGMENT_HOP_SECONDS):
        win=max(256,int(window_seconds*self.sr)); hop=max(128,int(hop_seconds*self.sr))
        if len(y)<win: y=np.pad(y,(0,win-len(y)))
        starts=list(range(0,max(1,len(y)-win+1),hop)) or [0]
        return torch.tensor(np.stack([self.one_segment(y[s:s+win]) for s in starts]),dtype=torch.float32)


class MusicStructureGraphBuilder:
    def __init__(self, top_k=GRAPH_TOP_K, min_similarity=GRAPH_MIN_SIM, add_temporal=True):
        self.top_k=int(top_k); self.min_similarity=float(min_similarity); self.add_temporal=bool(add_temporal)
    def build(self,x):
        n=int(x.shape[0]); edges=set()
        if n==1: edges.add((0,0))
        if self.add_temporal:
            for i in range(n-1): edges.add((i,i+1)); edges.add((i+1,i))
        if n>2 and self.top_k>0:
            xn=F.normalize(x.float(),p=2,dim=-1); sim=xn@xn.T; sim.fill_diagonal_(-1.0)
            k=min(self.top_k,n-1)
            for i in range(n):
                vals,idx=torch.topk(sim[i],k=k)
                for v,j in zip(vals.tolist(),idx.tolist()):
                    if v>=self.min_similarity: edges.add((i,j)); edges.add((j,i))
        if not edges: edges.add((0,0))
        edge_index=torch.tensor(sorted(edges),dtype=torch.long).T.contiguous()
        return Data(x=x,edge_index=edge_index)

extractor = AudioFeatureExtractor()
graph_builder = MusicStructureGraphBuilder()
log(f'Audio node feature dimension = {extractor.feature_dim}; graph top-k={GRAPH_TOP_K}, min cosine={GRAPH_MIN_SIM}.', 'MODEL')


In [ ]:
# Cell 5 — Build/resume ALL MusicCaps graphs and log graph statistics.
MC_GRAPH_DIR = ROOT/'data/processed/musiccaps_graphs'

def preprocess_one_musiccaps(item):
    ytid, audio_path = item
    out = MC_GRAPH_DIR/f'{ytid}.pt'
    if out.exists() and out.stat().st_size > 1024:
        try:
            g=torch.load(out,map_location='cpu',weights_only=False)
            return {'ytid':ytid,'status':'present','nodes':int(g.num_nodes),'edges':int(g.num_edges)}
        except Exception: pass
    try:
        y,_=librosa.load(str(audio_path),sr=AUDIO_SR,mono=True,duration=CLIP_SECONDS)
        x=extractor.extract_segments(y)
        g=graph_builder.build(x); g.ytid=ytid
        torch.save(g,out)
        return {'ytid':ytid,'status':'created','nodes':int(g.num_nodes),'edges':int(g.num_edges)}
    except Exception as e:
        return {'ytid':ytid,'status':'failed','error':repr(e)}

items=list(usable_audio.items())
log(f'Preprocessing/resuming {len(items):,} paired MusicCaps graphs with {NUM_WORKERS} workers.', 'PREPROCESS')
graph_log=[]
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    futs=[ex.submit(preprocess_one_musiccaps,it) for it in items]
    for i,f in enumerate(tqdm(as_completed(futs), total=len(futs), desc='MusicCaps graphs')):
        graph_log.append(f.result())
        if (i+1)%500==0:
            good=sum(r['status'] in ('present','created') for r in graph_log)
            log(f'Graph progress {i+1:,}/{len(futs):,}; usable={good:,}', 'PREPROCESS')
write_json(graph_log,'results/musiccaps_preprocess_full.json')

mc_graph_paths={p.stem:p for p in MC_GRAPH_DIR.glob('*.pt')}
df_mc = df_mc_raw[df_mc_raw.ytid.astype(str).isin(mc_graph_paths)].copy().reset_index(drop=True)
node_counts=[r.get('nodes',0) for r in graph_log if r.get('nodes')]
edge_counts=[r.get('edges',0) for r in graph_log if r.get('edges')]
log(f'Final paired MusicCaps set: {len(df_mc):,} rows / {len(mc_graph_paths):,} graphs.', 'DATA')
if node_counts:
    log(f'Graph nodes mean={np.mean(node_counts):.1f}, median={np.median(node_counts):.1f}; edges mean={np.mean(edge_counts):.1f}.', 'DATA')
assert len(df_mc)>0, 'No paired MusicCaps graphs are available. Check the download log before training.'


In [ ]:
# Cell 6 — Inspect MusicCaps tags, one real graph, and one feature heatmap.
from collections import Counter

def parse_aspects(s):
    try: vals=ast.literal_eval(str(s))
    except Exception: vals=[]
    return [str(t).strip().lower() for t in vals if len(str(t).strip())>2]

tag_counts=Counter(t for s in df_mc.aspect_list.dropna() for t in parse_aspects(s))
top_plot=tag_counts.most_common(25)
plt.figure(figsize=(11,7))
plt.barh([x[0] for x in top_plot][::-1],[x[1] for x in top_plot][::-1])
plt.title(f'MusicCaps aspect support — top 25 of {len(tag_counts):,} observed tags')
plt.xlabel('Number of paired clips'); plt.tight_layout(); save_current_figure('musiccaps_tag_support.png'); plt.show()

sample_id=str(df_mc.iloc[0].ytid)
sample_g=torch.load(mc_graph_paths[sample_id],map_location='cpu',weights_only=False)
G=nx.Graph(); G.add_nodes_from(range(sample_g.num_nodes)); G.add_edges_from(sample_g.edge_index.T.tolist())
plt.figure(figsize=(9,6)); pos=nx.spring_layout(G,seed=42)
nx.draw_networkx(G,pos,node_size=280,font_size=8,with_labels=True,width=0.8)
plt.title(f'Real MusicCaps structure graph: {sample_id} | nodes={sample_g.num_nodes}, edges={sample_g.num_edges}')
plt.axis('off'); plt.tight_layout(); save_current_figure('musiccaps_sample_graph.png'); plt.show()

plt.figure(figsize=(12,5));
plt.imshow(sample_g.x.numpy(),aspect='auto',interpolation='nearest')
plt.title(f'Node-feature matrix for {sample_id} ({sample_g.num_nodes} segments × {sample_g.x.shape[1]} features)')
plt.xlabel('Feature dimension'); plt.ylabel('Segment node'); plt.colorbar(label='Raw feature value'); plt.tight_layout(); save_current_figure('musiccaps_sample_feature_matrix.png'); plt.show()
print('[SAMPLE CAPTION]', df_mc.iloc[0].caption)
print('[SAMPLE ASPECTS]', df_mc.iloc[0].aspect_list)


In [ ]:
# Cell 7 — Multilabel-stratified split, train-only vocabulary, train-only graph scaling, token cache.
def aspects_multihot(strings, vocab):
    idx={t:i for i,t in enumerate(vocab)}; Y=np.zeros((len(strings),len(vocab)),dtype=np.int64)
    for r,s in enumerate(strings):
        for t in parse_aspects(s):
            if t in idx: Y[r,idx[t]]=1
    return Y

# Use labels only to create a balanced split; the FINAL supervised vocabulary is still built from training rows only.
strat_vocab=[t for t,c in tag_counts.most_common(120) if c>=3]
Y_strat=aspects_multihot(df_mc.aspect_list.tolist(),strat_vocab)
idx_all=np.arange(len(df_mc))
ms1=MultilabelStratifiedShuffleSplit(n_splits=1,test_size=0.30,random_state=SPLIT_SEED)
tr_idx,tmp_idx=next(ms1.split(idx_all,Y_strat))
ms2=MultilabelStratifiedShuffleSplit(n_splits=1,test_size=0.50,random_state=SPLIT_SEED)
va_rel,te_rel=next(ms2.split(tmp_idx,Y_strat[tmp_idx]))
va_idx,te_idx=tmp_idx[va_rel],tmp_idx[te_rel]
train_df=df_mc.iloc[tr_idx].copy().reset_index(drop=True)
val_df=df_mc.iloc[va_idx].copy().reset_index(drop=True)
test_df=df_mc.iloc[te_idx].copy().reset_index(drop=True)

train_counts=Counter(t for s in train_df.aspect_list for t in parse_aspects(s))
TAG_VOCAB=[t for t,c in train_counts.most_common() if c>=MIN_TAG_COUNT][:TOP_K_TAGS]
TAG2IDX={t:i for i,t in enumerate(TAG_VOCAB)}
assert len(TAG_VOCAB)>=10, 'Too few supported training tags; more paired MusicCaps audio is needed.'

split_payload={'train':train_df.ytid.astype(str).tolist(),'val':val_df.ytid.astype(str).tolist(),'test':test_df.ytid.astype(str).tolist()}
write_json(split_payload,'data/splits/musiccaps_split.json'); write_json(TAG_VOCAB,'data/splits/musiccaps_tag_vocab.json')
log(f'MusicCaps split: train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}; supervised tags={len(TAG_VOCAB)}.', 'SPLIT')
print('Top supervised tags:', TAG_VOCAB[:20])

class GraphFeatureScaler:
    def __init__(self,mean,std): self.mean=mean.float(); self.std=std.float().clamp_min(1e-5)
    @classmethod
    def fit(cls,paths):
        xs=[]
        for p in tqdm(paths,desc='Fitting graph scaler'):
            xs.append(torch.load(p,map_location='cpu',weights_only=False).x.float())
        x=torch.cat(xs,0); return cls(x.mean(0),x.std(0,unbiased=False))
    def transform(self,g):
        q=g.clone(); q.x=(q.x.float()-self.mean)/self.std; return q
    def state_dict(self): return {'mean':self.mean,'std':self.std}

train_graph_paths=[mc_graph_paths[str(x)] for x in train_df.ytid]
mc_scaler=GraphFeatureScaler.fit(train_graph_paths)
torch.save(mc_scaler.state_dict(),'data/processed/musiccaps_graph_scaler.pt')

log(f'Loading tokenizer once and pre-tokenizing {len(df_mc):,} captions to keep the GPU fed.', 'TEXT')
tokenizer=AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
tokens=tokenizer(df_mc.caption.astype(str).tolist(),padding='max_length',truncation=True,max_length=TEXT_MAX_LENGTH,return_tensors='pt')
token_cache={str(df_mc.iloc[i].ytid):(tokens['input_ids'][i],tokens['attention_mask'][i]) for i in range(len(df_mc))}

def tags_to_tensor(s):
    y=torch.zeros(len(TAG_VOCAB),dtype=torch.float32)
    for t in parse_aspects(s):
        if t in TAG2IDX: y[TAG2IDX[t]]=1.0
    return y

class MusicCapsDataset(Dataset):
    def __init__(self,df,scaler): self.df=df.reset_index(drop=True); self.scaler=scaler
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; ytid=str(r.ytid)
        g=torch.load(mc_graph_paths[ytid],map_location='cpu',weights_only=False); g=self.scaler.transform(g)
        ids,mask=token_cache[ytid]
        return g,ids,mask,tags_to_tensor(r.aspect_list),ytid,str(r.caption)

def collate_musiccaps(items):
    g,ids,mask,y,ytid,cap=zip(*items)
    return Batch.from_data_list(list(g)),torch.stack(ids),torch.stack(mask),torch.stack(y),list(ytid),list(cap)

train_ds=MusicCapsDataset(train_df,mc_scaler); val_ds=MusicCapsDataset(val_df,mc_scaler); test_ds=MusicCapsDataset(test_df,mc_scaler)

def make_mc_loader(ds,batch_size,shuffle):
    return DataLoader(ds,batch_size=batch_size,collate_fn=collate_musiccaps,drop_last=False,**safe_loader_kwargs(shuffle))
train_loader=make_mc_loader(train_ds,TRAIN_BS,True)
val_loader=make_mc_loader(val_ds,EVAL_BS,False)
test_loader=make_mc_loader(test_ds,EVAL_BS,False)
log(f'DataLoaders ready: train batches={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}.', 'DATA')


In [ ]:
# Cell 8 — Inline BERT, GraphSAGE, fusion, and classification models.
class GraphSAGEEncoder(nn.Module):
    def __init__(self,in_dim,hidden=GRAPH_HIDDEN,layers=4,dropout=0.20):
        super().__init__(); dims=[in_dim]+[hidden]*layers
        self.convs=nn.ModuleList([SAGEConv(dims[i],dims[i+1],aggr='mean') for i in range(layers)])
        self.norms=nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])
        self.dropout=dropout; self.out_dim=hidden
        self.input_proj=nn.Linear(in_dim,hidden) if in_dim!=hidden else nn.Identity()
    def forward(self,x,edge_index,batch):
        h=x
        for i,(conv,norm) in enumerate(zip(self.convs,self.norms)):
            z=F.gelu(norm(conv(h,edge_index)))
            if i>0 and h.shape[-1]==z.shape[-1]: z=z+h
            h=F.dropout(z,p=self.dropout,training=self.training)
        return global_mean_pool(h,batch)

class TextEncoder(nn.Module):
    def __init__(self,name=TEXT_MODEL_NAME,full_finetune=True):
        super().__init__(); self.backbone=AutoModel.from_pretrained(name); self.hidden_dim=int(self.backbone.config.hidden_size)
        if not full_finetune:
            for p in self.backbone.parameters(): p.requires_grad=False
    def forward(self,input_ids,attention_mask):
        out=self.backbone(input_ids=input_ids,attention_mask=attention_mask)
        H=out.last_hidden_state; cls=H[:,0]
        return H,cls,attention_mask.bool()

class EarlyConcatFusion(nn.Module):
    def __init__(self,dg,dt,d=FUSION_DIM,dropout=.2):
        super().__init__(); self.net=nn.Sequential(nn.Linear(dg+dt,d),nn.LayerNorm(d),nn.GELU(),nn.Dropout(dropout)); self.out_dim=d
    def forward(self,g,H,cls,mask): return self.net(torch.cat([g,cls],-1))

class CrossAttentionFusion(nn.Module):
    def __init__(self,dg,dt,d=FUSION_DIM,heads=ATTN_HEADS,dropout=.2):
        super().__init__(); self.q=nn.Linear(dg,d); self.k=nn.Linear(dt,d); self.v=nn.Linear(dt,d)
        self.attn=nn.MultiheadAttention(d,heads,dropout=dropout,batch_first=True)
        self.out=nn.Sequential(nn.Linear(dg+d,d),nn.LayerNorm(d),nn.GELU(),nn.Dropout(dropout)); self.out_dim=d
    def forward(self,g,H,cls,mask,return_attention=False):
        q=self.q(g).unsqueeze(1); k=self.k(H); v=self.v(H); pad=~mask.bool()
        ctx,w=self.attn(q,k,v,key_padding_mask=pad,need_weights=return_attention,average_attn_weights=False)
        z=self.out(torch.cat([g,ctx.squeeze(1)],-1))
        return (z,w) if return_attention else z

class GatedFusion(nn.Module):
    def __init__(self,dg,dt,d=FUSION_DIM,dropout=.2):
        super().__init__(); self.gp=nn.Linear(dg,d); self.tp=nn.Linear(dt,d); self.gate=nn.Linear(dg+dt,d)
        self.norm=nn.LayerNorm(d); self.drop=nn.Dropout(dropout); self.out_dim=d
    def forward(self,g,H,cls,mask):
        ga=torch.tanh(self.gp(g)); ta=torch.tanh(self.tp(cls)); gate=torch.sigmoid(self.gate(torch.cat([g,cls],-1)))
        return self.drop(self.norm(gate*ga+(1-gate)*ta))

class MLPHead(nn.Module):
    def __init__(self,d,nout,dropout=.2):
        super().__init__(); self.net=nn.Sequential(nn.Dropout(dropout),nn.Linear(d,d),nn.GELU(),nn.LayerNorm(d),nn.Dropout(dropout),nn.Linear(d,nout))
    def forward(self,z): return self.net(z)

class BertOnlyTagModel(nn.Module):
    def __init__(self,n_tags):
        super().__init__(); self.text=TextEncoder(full_finetune=FULL_TEXT_FINETUNE); self.proj=nn.Sequential(nn.Linear(self.text.hidden_dim,FUSION_DIM),nn.LayerNorm(FUSION_DIM),nn.GELU(),nn.Dropout(.2)); self.head=MLPHead(FUSION_DIM,n_tags)
    def forward(self,batch):
        H,cls,mask=self.text(batch['input_ids'],batch['attention_mask']); z=self.proj(cls); return self.head(z),z,None

class GraphOnlyTagModel(nn.Module):
    def __init__(self,in_dim,n_tags):
        super().__init__(); self.graph=GraphSAGEEncoder(in_dim); self.head=MLPHead(self.graph.out_dim,n_tags)
    def forward(self,batch):
        g=self.graph(batch['graph'].x,batch['graph'].edge_index,batch['graph'].batch); return self.head(g),g,g

class FusionTagModel(nn.Module):
    def __init__(self,in_dim,n_tags,kind):
        super().__init__(); self.kind=kind; self.graph=GraphSAGEEncoder(in_dim); self.text=TextEncoder(full_finetune=FULL_TEXT_FINETUNE)
        if kind=='early': self.fusion=EarlyConcatFusion(self.graph.out_dim,self.text.hidden_dim)
        elif kind=='cross_attention': self.fusion=CrossAttentionFusion(self.graph.out_dim,self.text.hidden_dim)
        elif kind=='gated': self.fusion=GatedFusion(self.graph.out_dim,self.text.hidden_dim)
        else: raise ValueError(kind)
        self.head=MLPHead(self.fusion.out_dim,n_tags)
    def forward(self,batch,return_attention=False):
        g=self.graph(batch['graph'].x,batch['graph'].edge_index,batch['graph'].batch)
        H,cls,mask=self.text(batch['input_ids'],batch['attention_mask'])
        if self.kind=='cross_attention' and return_attention:
            z,w=self.fusion(g,H,cls,mask,True); return self.head(z),z,g,w
        z=self.fusion(g,H,cls,mask); return self.head(z),z,g

IN_DIM=int(train_ds[0][0].x.shape[1])
log(f'Model definitions ready. Graph input dim={IN_DIM}; tags={len(TAG_VOCAB)}.', 'MODEL')


In [ ]:
# Cell 9 — Metrics, validation threshold calibration, AMP training, early stopping, detailed epoch logs.
def calibrate_thresholds(y_true,probs,lo=.10,hi=.90,step=.025):
    grid=np.arange(lo,hi+1e-9,step); th=np.full(y_true.shape[1],.5,dtype=np.float32)
    for k in range(y_true.shape[1]):
        if y_true[:,k].sum()==0: continue
        scores=[f1_score(y_true[:,k],probs[:,k]>=t,zero_division=0) for t in grid]
        th[k]=float(grid[int(np.argmax(scores))])
    return th

def multilabel_metrics(y_true,probs,thresholds):
    pred=probs>=np.asarray(thresholds)[None,:]
    out={'macro_f1':float(f1_score(y_true,pred,average='macro',zero_division=0)),
         'micro_f1':float(f1_score(y_true,pred,average='micro',zero_division=0))}
    valid=y_true.sum(0)>0
    out['macro_auprc']=float(average_precision_score(y_true[:,valid],probs[:,valid],average='macro')) if valid.any() else float('nan')
    return out

def pos_weight_from_ds(ds):
    ys=torch.stack([tags_to_tensor(s) for s in ds.df.aspect_list]); pos=ys.sum(0); neg=len(ys)-pos
    return (neg/(pos+1e-6)).clamp(1.0,30.0)


def move_mc_batch(raw):
    graph,ids,mask,y,ytids,caps=raw
    return {'graph':graph.to(DEVICE,non_blocking=True), 'input_ids':ids.to(DEVICE,non_blocking=True),
            'attention_mask':mask.to(DEVICE,non_blocking=True),'y':y.to(DEVICE,non_blocking=True),
            'ytids':ytids,'captions':caps}

@torch.no_grad()
def evaluate_tag_model(model,loader):
    model.eval(); probs=[]; ys=[]; zs=[]; gids=[]
    for raw in loader:
        b=move_mc_batch(raw)
        with autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=(DEVICE.type=='cuda')):
            out=model(b); logits,z,g=out[:3]
        probs.append(torch.sigmoid(logits).float().cpu()); ys.append(b['y'].float().cpu()); zs.append(z.float().cpu()); gids.extend(b['ytids'])
    return torch.cat(ys).numpy(),torch.cat(probs).numpy(),torch.cat(zs).numpy(),gids


def make_model(kind):
    if kind=='bert_only': return BertOnlyTagModel(len(TAG_VOCAB))
    if kind=='gnn_only': return GraphOnlyTagModel(IN_DIM,len(TAG_VOCAB))
    return FusionTagModel(IN_DIM,len(TAG_VOCAB),kind)


def optimizer_for(model):
    text_params=[]; other_params=[]
    for n,p in model.named_parameters():
        if not p.requires_grad: continue
        (text_params if '.text.backbone.' in n or n.startswith('text.backbone.') else other_params).append(p)
    groups=[]
    if other_params: groups.append({'params':other_params,'lr':8e-4})
    if text_params: groups.append({'params':text_params,'lr':2e-5})
    try: return torch.optim.AdamW(groups,weight_decay=1e-4,fused=(DEVICE.type=='cuda'))
    except TypeError: return torch.optim.AdamW(groups,weight_decay=1e-4)


def train_tag_model(kind,seed):
    seed_everything(seed)
    if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
    model=make_model(kind).to(DEVICE)
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad); total=sum(p.numel() for p in model.parameters())
    log(f'{kind} seed={seed}: trainable={trainable/1e6:.2f}M / total={total/1e6:.2f}M parameters.', 'TRAIN')
    opt=optimizer_for(model)
    total_steps=max(1,len(train_loader)*EPOCHS); warmup=max(1,int(.08*total_steps))
    sched=get_cosine_schedule_with_warmup(opt,warmup,total_steps)
    scaler=GradScaler('cuda',enabled=(DEVICE.type=='cuda' and AMP_DTYPE==torch.float16))
    pw=pos_weight_from_ds(train_ds).to(DEVICE)
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=pw)
    resume_path=ROOT/f'checkpoints/resume/musiccaps_{kind}_seed{seed}_best.pt'
    best_state=None; best_th=None; best_val=-1.; best_epoch=0; bad=0; history=[]; start_epoch=1
    if resume_path.exists():
        try:
            rr=torch.load(resume_path,map_location='cpu',weights_only=False)
            model.load_state_dict(rr['state_dict']); best_state=rr['state_dict']; best_th=np.asarray(rr['thresholds'],dtype=np.float32)
            best_val=float(rr.get('best_val',-1.)); best_epoch=int(rr.get('best_epoch',0)); history=list(rr.get('history',[])); start_epoch=best_epoch+1
            log(f'{kind} seed={seed}: warm-resuming from Drive best epoch {best_epoch} (val Macro-F1={best_val:.4f}).','RESUME')
        except Exception as e:
            log(f'Could not load resume checkpoint {resume_path}: {e}; starting from scratch.','WARN')
    for ep in range(start_epoch,EPOCHS+1):
        model.train(); run_loss=0.; seen=0; t0=time.time()
        for step,raw in enumerate(train_loader,1):
            b=move_mc_batch(raw); opt.zero_grad(set_to_none=True)
            with autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=(DEVICE.type=='cuda')):
                logits=model(b)[0]; loss=loss_fn(logits,b['y'])
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); sched.step()
            bs=b['y'].shape[0]; run_loss += float(loss.item())*bs; seen+=bs
            if step==1 or step%max(1,len(train_loader)//3)==0:
                log(f'{kind} s{seed} e{ep:02d} step {step:03d}/{len(train_loader):03d} loss={loss.item():.4f} | {gpu_mem_string()}', 'BATCH')
        yv,pv,_,_=evaluate_tag_model(model,val_loader)
        th=calibrate_thresholds(yv,pv); vm=multilabel_metrics(yv,pv,th)
        epoch_loss=run_loss/max(1,seen); elapsed=time.time()-t0; lr=opt.param_groups[0]['lr']
        history.append({'epoch':ep,'train_loss':epoch_loss,**{f'val_{k}':v for k,v in vm.items()},'lr':lr,'seconds':elapsed})
        log(f'{kind} s{seed} epoch {ep:02d}: loss={epoch_loss:.4f} val_macroF1={vm["macro_f1"]:.4f} val_microF1={vm["micro_f1"]:.4f} val_AUPRC={vm["macro_auprc"]:.4f} lr={lr:.2e} time={elapsed:.1f}s | {gpu_mem_string()}', 'EPOCH')
        if vm['macro_f1']>best_val+1e-5:
            best_val=vm['macro_f1']; best_epoch=ep; best_th=th.copy(); bad=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            # Persist each new validation best immediately. A Colab quota termination loses at most the current epoch.
            torch.save({'state_dict':best_state,'thresholds':best_th,'best_val':best_val,'best_epoch':best_epoch,'history':history,
                        'model_kind':kind,'seed':seed,'tag_vocab':TAG_VOCAB,'text_model':TEXT_MODEL_NAME},resume_path)
            log(f'Persisted new best to Drive: {resume_path.name}','CHECKPOINT')
        else:
            bad+=1
            if bad>=PATIENCE:
                log(f'{kind} s{seed}: early stop after {ep} epochs; best epoch={best_epoch}.','STOP'); break
    if best_state is None:
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        best_th=np.full(len(TAG_VOCAB),0.5,dtype=np.float32)
    model.load_state_dict(best_state); model.to(DEVICE)
    yv,pv,zv,vids=evaluate_tag_model(model,val_loader)
    yt,pt,zt,tids=evaluate_tag_model(model,test_loader)
    vm=multilabel_metrics(yv,pv,best_th); tm=multilabel_metrics(yt,pt,best_th)
    log(f'{kind} s{seed} FINAL test: macroF1={tm["macro_f1"]:.4f} microF1={tm["micro_f1"]:.4f} AUPRC={tm["macro_auprc"]:.4f}', 'TEST')
    result={'kind':kind,'seed':seed,'state':best_state,'thresholds':best_th,'history':history,
            'val_metrics':vm,'test_metrics':tm,'val_probs':pv,'val_y':yv,'test_probs':pt,'test_y':yt,
            'val_z':zv,'test_z':zt,'val_ids':vids,'test_ids':tids,'best_epoch':best_epoch}
    del model
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return result

log('Training utilities ready. Validation—not test—is used for checkpoint selection and threshold calibration.', 'TRAIN')


In [ ]:
# Cell 10 — Train the full ablation suite on ALL available paired MusicCaps data, across 3 seeds.
# This is the main GPU-heavy cell. Full DistilBERT fine-tuning + large batches should use substantially more VRAM than the old cached/frozen pipeline.
MODEL_KINDS=['bert_only','gnn_only','early','cross_attention','gated']
all_runs={k:[] for k in MODEL_KINDS}
best_checkpoints={}
summary_rows=[]

for kind in MODEL_KINDS:
    log('='*78, 'TRAIN'); log(f'STARTING MODEL FAMILY: {kind}', 'TRAIN'); log('='*78, 'TRAIN')
    for seed in SEEDS:
        run=train_tag_model(kind,seed)
        run_state=run.pop('state')  # large tensor payload: retain only for the validation-best seed below
        summary_rows.append({'model':kind,'seed':seed,**run['val_metrics'],**{f'test_{k}':v for k,v in run['test_metrics'].items()},'best_epoch':run['best_epoch']})
        all_runs[kind].append(run)
        # Keep only the strongest seed by validation Macro-F1 to avoid test-based selection.
        cur=best_checkpoints.get(kind)
        if cur is None or run['val_metrics']['macro_f1']>cur['val_metrics']['macro_f1']:
            best_checkpoints[kind]={**run,'state':run_state}
        del run_state
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
    best=best_checkpoints[kind]
    torch.save({'state_dict':best['state'],'thresholds':best['thresholds'],'tag_vocab':TAG_VOCAB,'model_kind':kind,
                'seed':best['seed'],'split_seed':SPLIT_SEED,'graph_scaler':mc_scaler.state_dict(),'text_model':TEXT_MODEL_NAME,
                'full_text_finetune':FULL_TEXT_FINETUNE}, ROOT/f'checkpoints/musiccaps_{kind}_best.pt')
    log(f'Saved validation-selected {kind} checkpoint from seed {best["seed"]}.', 'CHECKPOINT')

summary_df=pd.DataFrame(summary_rows)
display(summary_df.round(4))
summary_df.to_csv('results/musiccaps_all_seed_metrics.csv',index=False)

# Aggregate seed metrics.
agg=[]
for kind in MODEL_KINDS:
    rows=[r['test_metrics'] for r in all_runs[kind]]
    vals=[r['val_metrics'] for r in all_runs[kind]]
    agg.append({'model':kind,
                'val_macro_f1_mean':np.mean([x['macro_f1'] for x in vals]),'val_macro_f1_std':np.std([x['macro_f1'] for x in vals]),
                'test_macro_f1_mean':np.mean([x['macro_f1'] for x in rows]),'test_macro_f1_std':np.std([x['macro_f1'] for x in rows]),
                'test_micro_f1_mean':np.mean([x['micro_f1'] for x in rows]),'test_micro_f1_std':np.std([x['micro_f1'] for x in rows]),
                'test_macro_auprc_mean':np.mean([x['macro_auprc'] for x in rows]),'test_macro_auprc_std':np.std([x['macro_auprc'] for x in rows])})
agg_df=pd.DataFrame(agg).sort_values('val_macro_f1_mean',ascending=False).reset_index(drop=True)
display(agg_df.round(4)); agg_df.to_csv('results/musiccaps_multiseed_aggregate.csv',index=False)
write_json({'rows':agg},'results/musiccaps_multiseed_metrics.json')

plt.figure(figsize=(10,5));
plt.bar(agg_df.model,agg_df.test_macro_f1_mean,yerr=agg_df.test_macro_f1_std,capsize=4)
plt.ylabel('Test Macro-F1'); plt.title('MusicCaps full paired-data ablation (mean ± std across seeds)'); plt.xticks(rotation=25); plt.tight_layout(); save_current_figure('musiccaps_ablation_macrof1.png'); plt.show()


In [ ]:
# Cell 11 — Validation-weighted ensemble, training curves, per-tag PR curves, t-SNE, and cross-attention grounding.
# Candidate ensembles are compared ONLY on validation Macro-F1. Test is evaluated once after selection.
# Including BERT-only is important here because the observed run showed it was the strongest individual architecture.
ensemble_candidates={
    'fusion_only':['early','cross_attention','gated'],
    'bert_plus_fusion':['bert_only','early','cross_attention','gated'],
    'all_models':['bert_only','gnn_only','early','cross_attention','gated'],
}
ensemble_trials=[]
for ens_name,ensemble_kinds in ensemble_candidates.items():
    val_weights=np.array([max(1e-6,np.mean([r['val_metrics']['macro_f1'] for r in all_runs[k]])) for k in ensemble_kinds],dtype=float)
    val_weights/=val_weights.sum()
    val_arch=[np.mean([r['val_probs'] for r in all_runs[k]],axis=0) for k in ensemble_kinds]
    test_arch=[np.mean([r['test_probs'] for r in all_runs[k]],axis=0) for k in ensemble_kinds]
    val_ens=sum(w*p for w,p in zip(val_weights,val_arch)); test_ens_candidate=sum(w*p for w,p in zip(val_weights,test_arch))
    yv=all_runs[ensemble_kinds[0]][0]['val_y']; yt=all_runs[ensemble_kinds[0]][0]['test_y']
    th=calibrate_thresholds(yv,val_ens); vm=multilabel_metrics(yv,val_ens,th)
    ensemble_trials.append({'name':ens_name,'kinds':ensemble_kinds,'weights':val_weights,'val_probs':val_ens,
                            'test_probs':test_ens_candidate,'thresholds':th,'validation':vm})
best_ens=max(ensemble_trials,key=lambda x:x['validation']['macro_f1'])
test_ens=best_ens['test_probs']; ens_th=best_ens['thresholds']; ens_val=best_ens['validation']; ens_test=multilabel_metrics(yt,test_ens,ens_th)
val_weights=best_ens['weights']; ensemble_kinds=best_ens['kinds']
print('Validation-selected ensemble:',best_ens['name'])
print('Validation-derived weights:',dict(zip(ensemble_kinds,val_weights.round(3))))
print('ENSEMBLE validation:',ens_val); print('ENSEMBLE test:',ens_test)
write_json({'selected_name':best_ens['name'],'members':ensemble_kinds,'weights':dict(zip(ensemble_kinds,val_weights.tolist())),
            'validation':ens_val,'test':ens_test,'thresholds':ens_th.tolist()},'results/musiccaps_fusion_ensemble.json')

# Training curves for validation-selected seed of each model.
plt.figure(figsize=(11,6))
for k in MODEL_KINDS:
    h=best_checkpoints[k]['history']; plt.plot([x['epoch'] for x in h],[x['val_macro_f1'] for x in h],label=k)
plt.xlabel('Epoch'); plt.ylabel('Validation Macro-F1'); plt.title('MusicCaps validation learning curves'); plt.grid(alpha=.25); plt.legend(); plt.tight_layout(); save_current_figure('musiccaps_validation_learning_curves.png'); plt.show()

# Per-tag PR curves for the ensemble (10 most-supported test tags).
support=yt.sum(0); top_idx=np.argsort(-support)[:10]
plt.figure(figsize=(10,7))
for k in top_idx:
    p,r,_=precision_recall_curve(yt[:,k],test_ens[:,k]); ap=average_precision_score(yt[:,k],test_ens[:,k])
    plt.plot(r,p,label=f'{TAG_VOCAB[k]} AP={ap:.2f}')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Ensemble precision-recall curves — 10 most-supported test tags'); plt.legend(fontsize=8); plt.grid(alpha=.2); plt.tight_layout(); save_current_figure('musiccaps_precision_recall_top10.png'); plt.show()

# t-SNE on latent z from validation-selected cross-attention checkpoint.
best_ca=best_checkpoints['cross_attention']; Z=best_ca['test_z']; n=len(Z)
if n>=10:
    perp=min(30,max(5,(n-1)//5)); Z2=TSNE(n_components=2,perplexity=perp,random_state=SPLIT_SEED,init='pca',learning_rate='auto').fit_transform(Z)
    # Color by the first active target among top tags, or -1 if none.
    yca=best_ca['test_y']; color=np.argmax(yca,axis=1); has=yca.sum(1)>0; color=np.where(has,color,-1)
    plt.figure(figsize=(9,7)); sc=plt.scatter(Z2[:,0],Z2[:,1],c=color,s=28,alpha=.8); plt.title('t-SNE of cross-attention fused test embeddings'); plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2'); plt.tight_layout(); save_current_figure('musiccaps_tsne_cross_attention.png'); plt.show()

# Attention grounding on one held-out caption.
model=make_model('cross_attention').to(DEVICE); model.load_state_dict(best_ca['state']); model.eval(); raw=next(iter(test_loader)); b=move_mc_batch(raw)
with torch.no_grad(), autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=(DEVICE.type=='cuda')):
    logits,z,g,w=model(b,return_attention=True)
att=w[0].float().cpu().mean(0).squeeze(0).numpy()  # average heads
ids=b['input_ids'][0].cpu().tolist(); mask=b['attention_mask'][0].cpu().bool().numpy(); toks=tokenizer.convert_ids_to_tokens(ids)
valid_n=int(mask.sum()); att=att[:valid_n]; toks=toks[:valid_n]
plt.figure(figsize=(max(10,valid_n*.35),2.6)); plt.imshow(att[None,:],aspect='auto'); plt.yticks([]); plt.xticks(range(valid_n),toks,rotation=60,ha='right',fontsize=8); plt.colorbar(label='Cross-attention weight'); plt.title('Cross-attention over one held-out MusicCaps caption'); plt.tight_layout(); save_current_figure('musiccaps_cross_attention_tokens.png'); plt.show()
example_caption=b['captions'][0]
del model, b
if DEVICE.type=='cuda': torch.cuda.empty_cache()
print('Caption:',example_caption)


## MusicCaps Task 4 — true-pair end-to-end contrastive retrieval

Unlike the earlier low-memory shortcut that first cached frozen BERT/GNN embeddings, this section trains the graph encoder, DistilBERT, and projection heads together with symmetric InfoNCE. A large in-batch negative set is used to make the GPU do useful work and to strengthen audio–caption alignment.


In [ ]:
# Cell 12 — End-to-end contrastive GraphSAGE + DistilBERT with full paired data and retrieval plots.
class ContrastiveModel(nn.Module):
    def __init__(self,in_dim,embed=256,temperature=.07):
        super().__init__(); self.graph=GraphSAGEEncoder(in_dim); self.text=TextEncoder(full_finetune=FULL_TEXT_FINETUNE)
        self.audio_proj=nn.Sequential(nn.Linear(self.graph.out_dim,embed),nn.GELU(),nn.Linear(embed,embed))
        self.text_proj=nn.Sequential(nn.Linear(self.text.hidden_dim,embed),nn.GELU(),nn.Linear(embed,embed))
        self.logit_scale=nn.Parameter(torch.tensor(math.log(1/temperature),dtype=torch.float32))
    def forward(self,b):
        g=self.graph(b['graph'].x,b['graph'].edge_index,b['graph'].batch)
        H,cls,mask=self.text(b['input_ids'],b['attention_mask'])
        za=F.normalize(self.audio_proj(g),dim=-1); zt=F.normalize(self.text_proj(cls),dim=-1)
        return za,zt
    def loss(self,za,zt):
        scale=self.logit_scale.exp().clamp(max=100); logits=scale*(za@zt.T); y=torch.arange(len(za),device=za.device)
        return .5*(F.cross_entropy(logits,y)+F.cross_entropy(logits.T,y))

def retrieval_metrics(za,zt,ks=(1,5,10)):
    sim=zt@za.T; n=sim.shape[0]; target=torch.arange(n,device=sim.device)[:,None]; out={}
    r=sim.argsort(1,descending=True)
    for k in ks:
        if k<=n: out[f'caption_to_audio_R@{k}']=float((r[:,:k]==target).any(1).float().mean())
    r2=sim.T.argsort(1,descending=True)
    for k in ks:
        if k<=n: out[f'audio_to_caption_R@{k}']=float((r2[:,:k]==target).any(1).float().mean())
    return out

contrast_train=make_mc_loader(train_ds,CONTRASTIVE_BS,True)
contrast_val=make_mc_loader(val_ds,EVAL_BS,False)
contrast_test=make_mc_loader(test_ds,EVAL_BS,False)
seed_everything(SEEDS[0]); cm=ContrastiveModel(IN_DIM).to(DEVICE); opt=optimizer_for(cm)
steps=len(contrast_train)*30; sched=get_cosine_schedule_with_warmup(opt,max(1,int(.08*steps)),steps)
scaler=GradScaler('cuda',enabled=(DEVICE.type=='cuda' and AMP_DTYPE==torch.float16))
best_state=None; best_r5=-1.; bad=0; c_hist=[]

@torch.no_grad()
def encode_contrastive(model,loader):
    model.eval(); A=[]; T=[]; ids=[]
    for raw in loader:
        b=move_mc_batch(raw)
        with autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=(DEVICE.type=='cuda')): za,zt=model(b)
        A.append(za.float().cpu()); T.append(zt.float().cpu()); ids.extend(b['ytids'])
    return torch.cat(A).to(DEVICE),torch.cat(T).to(DEVICE),ids

for ep in range(1,31):
    cm.train(); tot=0.; seen=0; t0=time.time()
    for step,raw in enumerate(contrast_train,1):
        b=move_mc_batch(raw); opt.zero_grad(set_to_none=True)
        with autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=(DEVICE.type=='cuda')):
            za,zt=cm(b); loss=cm.loss(za,zt)
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(cm.parameters(),1.0); scaler.step(opt); scaler.update(); sched.step()
        tot += float(loss.item())*len(za); seen += len(za)
        if step==1 or step%max(1,len(contrast_train)//3)==0: log(f'contrast e{ep:02d} {step}/{len(contrast_train)} loss={loss.item():.4f} | {gpu_mem_string()}','BATCH')
    va,vt,_=encode_contrastive(cm,contrast_val); vm=retrieval_metrics(va,vt); r5=vm.get('caption_to_audio_R@5',0.0); el=time.time()-t0
    c_hist.append({'epoch':ep,'loss':tot/max(1,seen),**vm})
    log(f'contrast epoch {ep:02d}: loss={tot/max(1,seen):.4f} val C→A R@5={r5:.4f} time={el:.1f}s | {gpu_mem_string()}','EPOCH')
    if r5>best_r5+1e-6:
        best_r5=r5; bad=0; best_state={k:v.detach().cpu().clone() for k,v in cm.state_dict().items()}
    else:
        bad+=1
        if bad>=6: log('Contrastive early stop.','STOP'); break
cm.load_state_dict(best_state); cm.to(DEVICE)
ta,tt,test_ids=encode_contrastive(cm,contrast_test); retrieval=retrieval_metrics(ta,tt)
print('Held-out retrieval:',json.dumps(retrieval,indent=2)); write_json(retrieval,'results/musiccaps_retrieval.json')
torch.save({'state_dict':best_state,'text_model':TEXT_MODEL_NAME,'graph_scaler':mc_scaler.state_dict(),'metrics':retrieval},'checkpoints/musiccaps_contrastive_best.pt')

plt.figure(figsize=(9,5)); plt.plot([x['epoch'] for x in c_hist],[x['loss'] for x in c_hist],marker='o'); plt.xlabel('Epoch'); plt.ylabel('InfoNCE loss'); plt.title('MusicCaps contrastive training loss'); plt.grid(alpha=.25); plt.tight_layout(); save_current_figure('musiccaps_contrastive_loss.png'); plt.show()
plt.figure(figsize=(9,5)); plt.plot([x['epoch'] for x in c_hist],[x.get('caption_to_audio_R@5',np.nan) for x in c_hist],marker='o'); plt.xlabel('Epoch'); plt.ylabel('Validation R@5'); plt.title('Caption → audio retrieval during training'); plt.grid(alpha=.25); plt.tight_layout(); save_current_figure('musiccaps_contrastive_val_r5.png'); plt.show()

# Similarity heatmap for a manageable held-out slice; metrics above were computed on the ENTIRE test set.
N_HEAT=min(30,len(ta)); sim=(tt[:N_HEAT]@ta[:N_HEAT].T).float().cpu().numpy()
plt.figure(figsize=(8,7)); plt.imshow(sim,aspect='auto'); plt.colorbar(label='Cosine similarity'); plt.xlabel('Audio graph index'); plt.ylabel('Caption index'); plt.title(f'Held-out cross-modal similarity matrix (first {N_HEAT}; diagonal = true pair)'); plt.tight_layout(); save_current_figure('musiccaps_retrieval_similarity_heatmap.png'); plt.show()


## GTZAN Task 2 — full 1,000-track audio-only GraphSAGE vs CNN baseline

This is deliberately kept separate from MusicCaps so no unrelated caption/audio pairing is introduced. The split is done at the original track level before training or evaluation.


In [ ]:
# Cell 13 — Download, preprocess, and visualize the complete GTZAN archive.
GTZAN_ROOT=ROOT/'data/raw/gtzan/genres'
GTZAN_ARCHIVE=ROOT/'data/raw/gtzan/genres.tar.gz'
if not GTZAN_ROOT.exists() or len(list(GTZAN_ROOT.glob('*/*'))) < 900:
    url='https://huggingface.co/datasets/marsyas/gtzan/resolve/main/data/genres.tar.gz'
    log(f'Downloading complete GTZAN archive: {url}','DOWNLOAD')
    urllib.request.urlretrieve(url,GTZAN_ARCHIVE)
    with tarfile.open(GTZAN_ARCHIVE,'r:gz') as tf:
        # Trusted public dataset archive; extract into data/raw/gtzan.
        tf.extractall(ROOT/'data/raw/gtzan')

gtzan_files=sorted([p for p in GTZAN_ROOT.glob('*/*') if p.suffix.lower() in {'.wav','.au','.mp3','.flac'} and not p.name.startswith('._')])
log(f'GTZAN source tracks discovered: {len(gtzan_files):,}.','DATA')
GT_GRAPH_DIR=ROOT/'data/processed/gtzan_graphs'; GT_MEL_DIR=ROOT/'data/processed/gtzan_mels'

def preprocess_gtzan_one(p):
    track_id=f'{p.parent.name}__{p.stem}'; gp=GT_GRAPH_DIR/f'{track_id}.pt'; mp=GT_MEL_DIR/f'{track_id}.npy'
    if gp.exists() and mp.exists(): return {'track_id':track_id,'genre':p.parent.name,'status':'present'}
    try:
        y,_=librosa.load(str(p),sr=AUDIO_SR,mono=True)
        g=graph_builder.build(extractor.extract_segments(y)); torch.save(g,gp)
        mel=librosa.feature.melspectrogram(y=y,sr=AUDIO_SR,n_mels=128,hop_length=512,power=2.0)
        mel=librosa.power_to_db(mel+1e-10,ref=np.max).astype(np.float32)
        target=1292
        if mel.shape[1]<target: mel=np.pad(mel,((0,0),(0,target-mel.shape[1])),constant_values=float(mel.min()))
        else: mel=mel[:,:target]
        np.save(mp,mel)
        return {'track_id':track_id,'genre':p.parent.name,'status':'created'}
    except Exception as e: return {'track_id':track_id,'genre':p.parent.name,'status':'failed','error':repr(e)}

rows=[]
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    futs=[ex.submit(preprocess_gtzan_one,p) for p in gtzan_files]
    for f in tqdm(as_completed(futs),total=len(futs),desc='GTZAN preprocess'): rows.append(f.result())
gtzan_index=pd.DataFrame([r for r in rows if r['status'] in ('present','created')]); gtzan_index.to_csv('data/processed/gtzan_index.csv',index=False)
log(f'GTZAN processed tracks: {len(gtzan_index):,}; failures={sum(r["status"]=="failed" for r in rows):,}.','DATA')

plt.figure(figsize=(10,4)); gtzan_index.genre.value_counts().sort_index().plot(kind='bar'); plt.ylabel('Tracks'); plt.title('GTZAN genre distribution'); plt.tight_layout(); save_current_figure('gtzan_genre_distribution.png'); plt.show()
if len(gtzan_index):
    rr=gtzan_index.iloc[0]; mel=np.load(GT_MEL_DIR/f'{rr.track_id}.npy')
    plt.figure(figsize=(12,4)); plt.imshow(mel,origin='lower',aspect='auto'); plt.colorbar(label='dB'); plt.title(f'GTZAN log-mel example: {rr.track_id}'); plt.xlabel('Frame'); plt.ylabel('Mel bin'); plt.tight_layout(); save_current_figure('gtzan_sample_logmel.png'); plt.show()


In [ ]:
# Cell 14 — Full GTZAN GraphSAGE and stronger mel-CNN training with large batches, logs, and confusion matrices.
genres=sorted(gtzan_index.genre.unique()); genre2idx={g:i for i,g in enumerate(genres)}
tr_df,rest_df=train_test_split(gtzan_index,test_size=.30,random_state=SPLIT_SEED,stratify=gtzan_index.genre)
va_df,te_df=train_test_split(rest_df,test_size=.50,random_state=SPLIT_SEED,stratify=rest_df.genre)
log(f'GTZAN split: train={len(tr_df)}, val={len(va_df)}, test={len(te_df)}; genres={genres}','SPLIT')

gt_scaler=GraphFeatureScaler.fit([GT_GRAPH_DIR/f'{x}.pt' for x in tr_df.track_id])

class GTGraphDs(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; g=torch.load(GT_GRAPH_DIR/f'{r.track_id}.pt',map_location='cpu',weights_only=False); g=gt_scaler.transform(g)
        return g,genre2idx[r.genre],r.track_id

def gt_graph_collate(items):
    g,y,ids=zip(*items); return Batch.from_data_list(list(g)),torch.tensor(y,dtype=torch.long),list(ids)

class MelDs(Dataset):
    def __init__(self,df,augment=False): self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; x=np.load(GT_MEL_DIR/f'{r.track_id}.npy').astype(np.float32); x=(x-x.mean())/(x.std()+1e-6)
        t=torch.tensor(x[None],dtype=torch.float32)
        if self.augment:
            # Lightweight SpecAugment: train only.
            if random.random()<.7:
                f0=random.randint(0,115); fw=random.randint(4,12); t[:,f0:min(128,f0+fw),:]=0
            if random.random()<.7:
                t0=random.randint(0,1200); tw=random.randint(20,80); t[:,:,t0:min(t.shape[-1],t0+tw)]=0
        return t,genre2idx[r.genre],r.track_id

class GraphGenreModel(nn.Module):
    def __init__(self): super().__init__(); self.graph=GraphSAGEEncoder(IN_DIM,hidden=max(256,GRAPH_HIDDEN//2),layers=4,dropout=.2); self.head=nn.Linear(self.graph.out_dim,len(genres))
    def forward(self,b): g=self.graph(b.x,b.edge_index,b.batch); return self.head(g)

class MelCNN(nn.Module):
    def __init__(self,nc):
        super().__init__(); ch=[1,32,64,128,192]
        blocks=[]
        for a,b in zip(ch[:-1],ch[1:]):
            blocks += [nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.GELU(),nn.MaxPool2d(2),nn.Dropout2d(.08)]
        self.f=nn.Sequential(*blocks,nn.AdaptiveAvgPool2d((1,1))); self.h=nn.Sequential(nn.Flatten(),nn.Dropout(.25),nn.Linear(ch[-1],nc))
    def forward(self,x): return self.h(self.f(x))

def make_gt_loaders():
    gtr=DataLoader(GTGraphDs(tr_df),batch_size=GRAPH_BATCH,collate_fn=gt_graph_collate,**safe_loader_kwargs(True))
    gva=DataLoader(GTGraphDs(va_df),batch_size=GRAPH_BATCH*2,collate_fn=gt_graph_collate,**safe_loader_kwargs(False))
    gte=DataLoader(GTGraphDs(te_df),batch_size=GRAPH_BATCH*2,collate_fn=gt_graph_collate,**safe_loader_kwargs(False))
    ctr=DataLoader(MelDs(tr_df,True),batch_size=CNN_BATCH,**safe_loader_kwargs(True))
    cva=DataLoader(MelDs(va_df,False),batch_size=CNN_BATCH*2,**safe_loader_kwargs(False))
    cte=DataLoader(MelDs(te_df,False),batch_size=CNN_BATCH*2,**safe_loader_kwargs(False))
    return gtr,gva,gte,ctr,cva,cte

gtr,gva,gte,ctr,cva,cte=make_gt_loaders()

@torch.no_grad()
def eval_gt(model,loader,is_graph):
    model.eval(); yy=[]; pp=[]
    for batch in loader:
        if is_graph:
            x,y,_=batch; logits=model(x.to(DEVICE)); y=y.to(DEVICE)
        else:
            x,y,_=batch; logits=model(x.to(DEVICE,non_blocking=True)); y=y.to(DEVICE,non_blocking=True)
        pp.extend(logits.argmax(1).cpu().tolist()); yy.extend(y.cpu().tolist())
    return {'accuracy':accuracy_score(yy,pp),'macro_f1':f1_score(yy,pp,average='macro')},np.array(yy),np.array(pp)

def train_gt(model,tr,va,is_graph,name,seed,epochs=50):
    seed_everything(seed); model=model.to(DEVICE)
    try: opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4,fused=(DEVICE.type=='cuda'))
    except TypeError: opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    scaler=GradScaler('cuda',enabled=(DEVICE.type=='cuda' and AMP_DTYPE==torch.float16)); best=None; bestm=-1; bad=0; hist=[]
    for ep in range(1,epochs+1):
        model.train(); tot=0.; n=0; t0=time.time()
        for batch in tr:
            opt.zero_grad(set_to_none=True)
            if is_graph:
                x,y,_=batch; x=x.to(DEVICE); y=y.to(DEVICE)
            else:
                x,y,_=batch; x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            with autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=(DEVICE.type=='cuda')):
                logits=model(x); loss=F.cross_entropy(logits,y,label_smoothing=.03)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update()
            tot+=float(loss.item())*len(y); n+=len(y)
        vm,_,_=eval_gt(model,va,is_graph); hist.append({'epoch':ep,'loss':tot/n,**vm})
        log(f'GTZAN {name} s{seed} e{ep:02d}: loss={tot/n:.4f} val_acc={vm["accuracy"]:.4f} val_macroF1={vm["macro_f1"]:.4f} time={time.time()-t0:.1f}s | {gpu_mem_string()}','EPOCH')
        if vm['macro_f1']>bestm+1e-5: bestm=vm['macro_f1']; bad=0; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            bad+=1
            if bad>=7: break
    model.load_state_dict(best); model.to(DEVICE); return model,hist

gt_results={'GNN':[],'CNN':[]}; gt_best={}
for seed in SEEDS:
    gm,gh=train_gt(GraphGenreModel(),gtr,gva,True,'GNN',seed); gmtr,yy,pp=eval_gt(gm,gte,True); gt_results['GNN'].append(gmtr)
    cm_,ch=train_gt(MelCNN(len(genres)),ctr,cva,False,'CNN',seed); cmtr,cyy,cpp=eval_gt(cm_,cte,False); gt_results['CNN'].append(cmtr)
    log(f'GTZAN seed={seed}: GNN test={gmtr} | CNN test={cmtr}','TEST')
    # Retain the best validation model of each family using last validation score from history.
    for name,model,h in [('GNN',gm,gh),('CNN',cm_,ch)]:
        score=max(x['macro_f1'] for x in h)
        if name not in gt_best or score>gt_best[name][0]: gt_best[name]=(score,{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},seed,h)

agg_gt={}
for name,rows in gt_results.items():
    agg_gt[name]={k:{'mean':float(np.mean([r[k] for r in rows])),'std':float(np.std([r[k] for r in rows]))} for k in rows[0]}
print(json.dumps(agg_gt,indent=2)); write_json({'seeds':gt_results,'aggregate':agg_gt},'results/gtzan_metrics.json')
torch.save({'state_dict':gt_best['GNN'][1],'genres':genres,'seed':gt_best['GNN'][2],'graph_scaler':gt_scaler.state_dict()},'checkpoints/gtzan_gnn_best.pt')
torch.save({'state_dict':gt_best['CNN'][1],'genres':genres,'seed':gt_best['CNN'][2]},'checkpoints/gtzan_cnn_best.pt')

# Final confusion matrices for validation-selected models.
gm=GraphGenreModel().to(DEVICE); gm.load_state_dict(gt_best['GNN'][1]); _,gy,gp=eval_gt(gm,gte,True)
cm_=MelCNN(len(genres)).to(DEVICE); cm_.load_state_dict(gt_best['CNN'][1]); _,cy,cp=eval_gt(cm_,cte,False)
for name,y,p in [('GraphSAGE',gy,gp),('Mel-CNN',cy,cp)]:
    fig,ax=plt.subplots(figsize=(9,8)); ConfusionMatrixDisplay(confusion_matrix(y,p),display_labels=genres).plot(ax=ax,xticks_rotation=45,colorbar=False); ax.set_title(f'GTZAN held-out confusion matrix — {name}'); plt.tight_layout(); save_current_figure(f'gtzan_confusion_{name.lower().replace("-","_")}.png'); plt.show()


In [ ]:
# Cell 15 — Comprehensive held-out MusicCaps classification export and qualitative predictions.
# Select between the best single architecture and the previously constructed ensemble using VALIDATION only.
best_arch=agg_df.iloc[0].model
selected=best_checkpoints[best_arch]
model=make_model(best_arch).to(DEVICE); model.load_state_dict(selected['state']); model.eval(); single_th=selected['thresholds']
yv_single,pv_single,_,_=evaluate_tag_model(model,val_loader)
yt_single,pt_single,_,test_ids=evaluate_tag_model(model,test_loader)
single_val=multilabel_metrics(yv_single,pv_single,single_th)
use_ensemble = ens_val['macro_f1'] > single_val['macro_f1'] + 1e-8
if use_ensemble:
    final_name=f"ensemble::{best_ens['name']}"; yt=yt; pt=test_ens; thresholds=ens_th
    log(f'Validation selected ensemble over single {best_arch}: {ens_val["macro_f1"]:.4f} vs {single_val["macro_f1"]:.4f}.','SELECT')
else:
    final_name=best_arch; yt=yt_single; pt=pt_single; thresholds=single_th
    log(f'Validation selected single {best_arch}: {single_val["macro_f1"]:.4f} >= ensemble {ens_val["macro_f1"]:.4f}.','SELECT')

del model
if DEVICE.type=='cuda': torch.cuda.empty_cache()

records=[]
row_lookup=test_df.set_index(test_df.ytid.astype(str))
for i,ytid in enumerate(test_ids):
    probs=pt[i]; truth=np.where(yt[i]>0.5)[0]; pred=np.where(probs>=thresholds)[0]
    records.append({'ytid':ytid,'caption':str(row_lookup.loc[ytid,'caption']) if ytid in row_lookup.index else '',
                    'true_tags':' | '.join(TAG_VOCAB[j] for j in truth),
                    'predicted_tags':' | '.join(TAG_VOCAB[j] for j in pred),
                    **{f'prob::{TAG_VOCAB[j]}':float(probs[j]) for j in range(len(TAG_VOCAB))}})
pred_df=pd.DataFrame(records); pred_df.to_csv(ROOT/'results/musiccaps_heldout_predictions_ALL.csv',index=False)
log(f'Exported ALL {len(pred_df):,} held-out classifications × {len(TAG_VOCAB)} tag probabilities.', 'EXPORT')

print(f'Validation-selected predictor: {final_name}')
print('Held-out metrics:',multilabel_metrics(yt,pt,thresholds))
display(pred_df[['ytid','caption','true_tags','predicted_tags']].head(30))

# Plot 10 qualitative probability profiles.
N_EX=min(10,len(pred_df))
fig,axes=plt.subplots(N_EX,1,figsize=(14,3*N_EX),squeeze=False)
for i in range(N_EX):
    top=np.argsort(-pt[i])[:10]; ax=axes[i,0]; ax.bar([TAG_VOCAB[j] for j in top],pt[i,top]); ax.axhline(float(np.mean(thresholds[top])),ls='--',alpha=.5); ax.set_ylim(0,1); ax.set_title(f'{test_ids[i]} — top probabilities'); ax.tick_params(axis='x',rotation=35)
plt.tight_layout(); save_current_figure('musiccaps_qualitative_probability_profiles.png'); plt.show()

In [ ]:
# Cell 16 — Final artifact inventory, metrics snapshot, and compressed output bundle.
# This does NOT fabricate missing metrics; it packages whatever this run actually produced.
artifacts=[]
for folder in ['checkpoints','results','data/splits']:
    for p in (ROOT/folder).rglob('*'):
        if p.is_file(): artifacts.append({'path':str(p.relative_to(ROOT)),'size_mb':p.stat().st_size/2**20})
artifact_df=pd.DataFrame(artifacts).sort_values('size_mb',ascending=False)
display(artifact_df.head(40).round({'size_mb':2}))
print(f'Total generated artifacts: {len(artifact_df):,}; total size={artifact_df.size_mb.sum():.1f} MB')

# Save a compact final metrics dashboard JSON.
final_dashboard={
    'musiccaps_available_pairs':len(df_mc),'musiccaps_tags':len(TAG_VOCAB),'musiccaps_split':{k:len(v) for k,v in split_payload.items()},
    'musiccaps_ablation':agg_df.to_dict(orient='records'),'musiccaps_fusion_ensemble':ens_test,
    'musiccaps_retrieval':retrieval,'gtzan':agg_gt,
    'runtime':{'device':str(DEVICE),'gpu':GPU_NAME,'vram_gb':VRAM_GB,'train_batch':TRAIN_BS,'contrastive_batch':CONTRASTIVE_BS,'full_text_finetune':FULL_TEXT_FINETUNE}
}
write_json(final_dashboard,'results/FINAL_DASHBOARD.json')

# Package trained weights + metrics + split definitions. Raw audio is deliberately excluded from the zip.
bundle_root=ROOT/'submission_runtime_artifacts';
if bundle_root.exists(): shutil.rmtree(bundle_root)
bundle_root.mkdir()
for folder in ['checkpoints','results','data/splits']:
    src=ROOT/folder; dst=bundle_root/folder; shutil.copytree(src,dst)
zip_path=shutil.make_archive(str(ROOT/'GNN_BERT_Music_Context_TRAINED_ARTIFACTS'),'zip',root_dir=bundle_root)
print('\n'+'='*96)
print('[DONE] Full inline training/evaluation pipeline completed.')
print('[DONE] Persistent Drive artifact bundle:',zip_path)
print('[DONE] Persistent project root:',ROOT)
print('[DONE] Key metrics: results/FINAL_DASHBOARD.json')
print('[DONE] Full held-out classification table: results/musiccaps_heldout_predictions_ALL.csv')
print('[DONE] Best weights: checkpoints/*.pt')
print('='*96)


## After this notebook finishes

Use `results/FINAL_DASHBOARD.json`, `results/musiccaps_multiseed_aggregate.csv`, `results/musiccaps_retrieval.json`, and `results/gtzan_metrics.json` as the numerical source for the final report. Keep the saved split JSON files and checkpoints with the submission so the reported test results are reproducible. The raw MusicCaps/GTZAN audio should normally **not** be placed inside the submission ZIP because of size/licensing; the processed graph examples, metrics, code/notebook, and trained weights are the relevant project artifacts.


All outputs now live under `MyDrive/CSE425_GNN_BERT_Music_Context/`. If Colab terminates, rerun from Cell 1: cached audio/graphs are reused, known failed YouTube IDs are skipped, and model training warm-resumes from validation-best Drive checkpoints.
